# ATS Resume Compliance Checker — Fine-Tuning Pipeline
Complete end-to-end pipeline: environment setup, dataset preparation, model loading,
LoRA fine-tuning, inference testing, and evaluation metrics.

**Instructions:** Upload the entire `colab/` folder to Google Drive (e.g., `My Drive/colab/`),
then run cells sequentially from top to bottom.

---
## 1. Environment Setup

In [1]:
# Mount Google Drive and set working directory
from google.colab import drive
drive.mount('/content/drive')

import os
COLAB_ROOT = '/content/drive/MyDrive/colab'  # Adjust if you uploaded elsewhere
os.chdir(COLAB_ROOT)
print(f"Working directory: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/colab


In [2]:
# Install dependencies (Colab already has PyTorch + CUDA)
!pip install -q transformers>=4.41.0 peft>=0.7.0 datasets>=2.16.0 accelerate>=0.25.0
!pip install -q bitsandbytes>=0.41.0 scipy sentencepiece protobuf
!pip install -q pyyaml tqdm matplotlib seaborn

In [3]:
# Verify installations and GPU
import torch
import transformers
import peft
import datasets
import accelerate

print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"PEFT: {peft.__version__}")
print(f"Datasets: {datasets.__version__}")
print(f"Accelerate: {accelerate.__version__}")

print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory: {gpu_mem:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
    if gpu_mem >= 16:
        print("[OK] Sufficient GPU memory for Phi-3 Mini with 4-bit quantization")
    elif gpu_mem >= 8:
        print("[WARNING] Limited GPU memory - 4-bit quantization required")
    else:
        print("[WARNING] Low GPU memory - consider a higher-tier Colab runtime")
else:
    print("[WARNING] No GPU detected - go to Runtime > Change runtime type > GPU")

try:
    import bitsandbytes as bnb
    print(f"\nbitsandbytes: {bnb.__version__} [OK]")
except ImportError:
    print("\n[ERROR] bitsandbytes not installed: !pip install bitsandbytes")

PyTorch: 2.10.0+cu128
Transformers: 5.0.0
PEFT: 0.18.1
Datasets: 4.0.0
Accelerate: 1.12.0

CUDA available: True
GPU: Tesla T4
GPU memory: 15.6 GB
CUDA version: 12.8
[WARNING] Limited GPU memory - 4-bit quantization required

bitsandbytes: 0.49.2 [OK]


In [4]:
# Verify folder structure
from pathlib import Path

for item in ["data", "configs", "data/raw_dataset.json",
             "configs/training_config.yaml", "configs/lora_config.yaml"]:
    status = "[OK]" if Path(item).exists() else "[MISSING]"
    print(f"  {status} {item}")

print("\nEnvironment setup complete!")

  [OK] data
  [OK] configs
  [OK] data/raw_dataset.json
  [OK] configs/training_config.yaml
  [OK] configs/lora_config.yaml

Environment setup complete!


---
## 2. Dataset Preparation

In [5]:
import json
import random
from collections import Counter

import yaml

# Load configuration
with open("configs/training_config.yaml", "r") as f:
    config = yaml.safe_load(f)

RAW_PATH = config["raw_dataset"]
TRAIN_PATH = config["train_dataset"]
VAL_PATH = config["validation_dataset"]
TRAIN_SPLIT = config["train_split"]
SEED = config["seed"]

print(f"Raw dataset: {RAW_PATH}")
print(f"Train split: {TRAIN_SPLIT}")
print(f"Seed: {SEED}")

Raw dataset: data/raw_dataset.json
Train split: 0.9
Seed: 42


In [6]:
# Load and validate raw dataset
with open(RAW_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data)} samples")
print(f"Sample keys: {list(raw_data[0].keys())}")

def validate_sample(sample):
    required_keys = {"instruction", "input", "output"}
    if not required_keys.issubset(sample.keys()):
        return False, "Missing required keys"
    try:
        output = json.loads(sample["output"])
    except (json.JSONDecodeError, TypeError):
        return False, "Invalid JSON in output"
    required_output_keys = {
        "ats_score", "score_breakdown", "matched_skills",
        "missing_skills", "weak_bullets", "formatting_issues",
        "overall_feedback"
    }
    if not required_output_keys.issubset(output.keys()):
        return False, f"Missing output keys: {required_output_keys - output.keys()}"
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        return False, "Invalid ATS score"
    return True, "Valid"

valid_samples = []
invalid_samples = []
for i, sample in enumerate(raw_data):
    is_valid, reason = validate_sample(sample)
    if is_valid:
        valid_samples.append(sample)
    else:
        invalid_samples.append((i, reason))

print(f"Valid: {len(valid_samples)} | Invalid: {len(invalid_samples)}")
for idx, reason in invalid_samples[:5]:
    print(f"  Sample {idx}: {reason}")

Loaded 200 samples
Sample keys: ['instruction', 'input', 'output']
Valid: 200 | Invalid: 0


In [7]:
# Score distribution
scores = [json.loads(s["output"])["ats_score"] for s in valid_samples]
print(f"ATS Scores — Min: {min(scores)}, Max: {max(scores)}, Mean: {sum(scores)/len(scores):.1f}")

buckets = Counter((s // 10) * 10 for s in scores)
for bucket in sorted(buckets):
    print(f"  {bucket:3d}-{bucket+9:3d}: {'#' * buckets[bucket]} ({buckets[bucket]})")

ATS Scores — Min: 35, Max: 92, Mean: 63.7
   30- 39: ############# (13)
   40- 49: ################################# (33)
   50- 59: ######################################### (41)
   60- 69: #################################### (36)
   70- 79: ####################################### (39)
   80- 89: ######################## (24)
   90- 99: ############## (14)


In [8]:
# Load tokenizer — must happen before format_prompt so we can use the correct EOS token
from transformers import AutoTokenizer

model_name = config["model_name"]
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    print(f"Tokenizer loaded: {model_name}")
except Exception:
    model_name = config["fallback_model"]
    print(f"Fallback to: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"EOS token: {repr(tokenizer.eos_token)} | Vocab: {tokenizer.vocab_size}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Tokenizer loaded: microsoft/phi-3-mini-4k-instruct
EOS token: '<|endoftext|>' | Vocab: 32000


In [9]:
# Format prompts, split, and save
# Use the actual tokenizer EOS token (Phi-3 uses <|end|>, not <|endoftext|>)
def format_prompt(sample, eos_token=None):
    token = eos_token if eos_token is not None else tokenizer.eos_token
    return (
        f"### Instruction:\n{sample['instruction']}\n\n"
        f"### Input:\n{sample['input']}\n\n"
        f"### Response:\n{sample['output']}{token}"
    )

for sample in valid_samples:
    sample["text"] = format_prompt(sample)

lengths = [len(s["text"]) for s in valid_samples]
print(f"EOS token: {repr(tokenizer.eos_token)}")
print(f"Prompt char lengths — Min: {min(lengths)}, Max: {max(lengths)}, Mean: {sum(lengths)/len(lengths):.0f}")

random.seed(SEED)
random.shuffle(valid_samples)
split_idx = int(len(valid_samples) * TRAIN_SPLIT)
train_data = valid_samples[:split_idx]
val_data = valid_samples[split_idx:]
print(f"Train: {len(train_data)} | Validation: {len(val_data)}")

os.makedirs(os.path.dirname(TRAIN_PATH), exist_ok=True)
with open(TRAIN_PATH, "w", encoding="utf-8") as f:
    json.dump(train_data, f, indent=2, ensure_ascii=False)
with open(VAL_PATH, "w", encoding="utf-8") as f:
    json.dump(val_data, f, indent=2, ensure_ascii=False)
print(f"Saved to {TRAIN_PATH} and {VAL_PATH}")

EOS token: '<|endoftext|>'
Prompt char lengths — Min: 2583, Max: 3797, Mean: 3155
Train: 180 | Validation: 20
Saved to data/train.json and data/validation.json


In [10]:
# Preview tokenization lengths on the training split
tokens = tokenizer(train_data[0]["text"], truncation=True, max_length=2048)
print(f"Sample token count: {len(tokens['input_ids'])} | Max seq length: {config['max_seq_length']}")

token_lengths = [len(tokenizer(s["text"], truncation=True, max_length=2048)["input_ids"]) for s in train_data[:50]]
print(f"Token lengths (first 50) — Min: {min(token_lengths)}, Max: {max(token_lengths)}, Mean: {sum(token_lengths)/len(token_lengths):.0f}")
print("\nDataset preparation complete!")

Sample token count: 909 | Max seq length: 2048
Token lengths (first 50) — Min: 801, Max: 1075, Mean: 931

Dataset preparation complete!


---
## 3. Model Loading & LoRA Setup

In [11]:
import yaml
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

with open("configs/training_config.yaml", "r") as f:
    train_cfg = yaml.safe_load(f)
with open("configs/lora_config.yaml", "r") as f:
    lora_cfg = yaml.safe_load(f)

print("Training Config:")
print(f"  Model: {train_cfg['model_name']} | Fallback: {train_cfg['fallback_model']}")
print(f"  4-bit: {train_cfg['use_4bit']} | Max seq: {train_cfg['max_seq_length']}")
print(f"\nLoRA Config:")
print(f"  r={lora_cfg['r']}, alpha={lora_cfg['lora_alpha']}, dropout={lora_cfg['lora_dropout']}")
print(f"  targets: {lora_cfg['target_modules']}, bias: {lora_cfg['bias']}")

Training Config:
  Model: microsoft/phi-3-mini-4k-instruct | Fallback: microsoft/phi-2
  4-bit: True | Max seq: 2048

LoRA Config:
  r=16, alpha=32, dropout=0.05
  targets: ['q_proj', 'v_proj', 'k_proj', 'o_proj'], bias: none


In [12]:
# Quantization config
use_4bit = train_cfg["use_4bit"] and torch.cuda.is_available()
bnb_config = None
if use_4bit:
    compute_dtype = getattr(torch, train_cfg.get("bnb_4bit_compute_dtype", "float16"))
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type=train_cfg.get("bnb_4bit_quant_type", "nf4"),
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=train_cfg.get("use_double_quant", True),
    )
    print("4-bit quantization configured (NF4 + double quant)")
else:
    print("Quantization disabled")

4-bit quantization configured (NF4 + double quant)


In [13]:
# Load tokenizer (trust_remote_code still needed for Phi-3 tokenizer)
model_name = train_cfg["model_name"]
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    print(f"Tokenizer loaded: {model_name}")
except Exception as e:
    print(f"Primary failed: {e}")
    model_name = train_cfg["fallback_model"]
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    print(f"Fallback tokenizer: {model_name}")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Vocab: {tokenizer.vocab_size} | Pad: {tokenizer.pad_token} | EOS: {tokenizer.eos_token}")

Tokenizer loaded: microsoft/phi-3-mini-4k-instruct
Vocab: 32000 | Pad: <|endoftext|> | EOS: <|endoftext|>


In [14]:
# Load base model
# transformers>=4.40 has native Phi-3 support — no trust_remote_code needed,
# which avoids the stale modeling_phi3.py with its DynamicCache.from_legacy_cache bug.

print(f"Loading base model: {model_name} ...")
model_kwargs = {
    "dtype": torch.float16,
    "device_map": "auto",
}
if bnb_config is not None:
    model_kwargs["quantization_config"] = bnb_config

model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
print(f"Loaded: {type(model).__name__} | Params: {sum(p.numel() for p in model.parameters()):,}")

Loading base model: microsoft/phi-3-mini-4k-instruct ...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Loaded: Phi3ForCausalLM | Params: 2,009,140,224


In [15]:
# Apply LoRA adapters
if train_cfg.get("gradient_checkpointing", True):
    model.gradient_checkpointing_enable()
    print("Gradient checkpointing enabled")

if bnb_config is not None:
    model = prepare_model_for_kbit_training(model)
    print("Prepared for k-bit training")

lora_config = LoraConfig(
    r=lora_cfg["r"],
    lora_alpha=lora_cfg["lora_alpha"],
    lora_dropout=lora_cfg["lora_dropout"],
    target_modules=lora_cfg["target_modules"],
    bias=lora_cfg["bias"],
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f"\nTrainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# Quick sanity check
test_input = tokenizer("### Instruction:\nEvaluate the resume", return_tensors="pt")
test_input = {k: v.to(model.device) for k, v in test_input.items()}
with torch.no_grad():
    output = model(**test_input)
print(f"Forward pass OK — logits shape: {output.logits.shape}")

Gradient checkpointing enabled
Prepared for k-bit training

Trainable: 3,145,728 / 3,824,225,280 (0.08%)
Forward pass OK — logits shape: torch.Size([1, 11, 32064])


---
## 4. Fine-Tuning

In [16]:
from datasets import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Load processed datasets
with open(train_cfg["train_dataset"], "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(train_cfg["validation_dataset"], "r", encoding="utf-8") as f:
    val_data = json.load(f)

assert "text" in train_data[0], "Run Section 2 first to prepare dataset with formatted prompts"
print(f"Train: {len(train_data)} | Validation: {len(val_data)}")

Train: 180 | Validation: 20


In [17]:
# Tokenize datasets — mask prompt tokens so loss is only on the response
max_seq_length = train_cfg.get("max_seq_length", 2048)
RESPONSE_TEMPLATE = "### Response:\n"

def tokenize_data(data, tokenizer, max_length):
    # Token IDs for the response template — used to find where response begins
    resp_ids = tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)
    resp_len = len(resp_ids)

    def tokenize_fn(examples):
        # No pre-padding; DataCollator will pad at batch time
        tokenized = tokenizer(
            examples["text"], truncation=True, max_length=max_length, padding=False
        )
        all_labels = []
        for input_ids in tokenized["input_ids"]:
            labels = list(input_ids)
            # Find "### Response:\n" and mask everything before it (prompt tokens)
            found = False
            for j in range(len(labels) - resp_len + 1):
                if labels[j : j + resp_len] == resp_ids:
                    for k in range(j + resp_len):
                        labels[k] = -100  # ignore prompt in loss
                    found = True
                    break
            if not found:
                # Shouldn't happen — mask everything as a safe fallback
                labels = [-100] * len(labels)
            all_labels.append(labels)
        tokenized["labels"] = all_labels
        return tokenized

    dataset = Dataset.from_dict({"text": [s["text"] for s in data]})
    return dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

print(f"Tokenizing with response-only labels (max_seq_length={max_seq_length})...")
train_dataset = tokenize_data(train_data, tokenizer, max_seq_length)
val_dataset = tokenize_data(val_data, tokenizer, max_seq_length)

# Verify masking is correct on first sample
first_labels = train_dataset[0]["labels"]
masked = sum(1 for l in first_labels if l == -100)
total_tok = len(first_labels)
print(f"Sample 0: {masked}/{total_tok} prompt tokens masked, {total_tok - masked} response tokens trained")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

Tokenizing with response-only labels (max_seq_length=2048)...


Map:   0%|          | 0/180 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Sample 0: 909/909 prompt tokens masked, 0 response tokens trained
Train: 180 | Val: 20


In [18]:
# Configure and launch training
from dataclasses import dataclass
from typing import Any, Dict, List

output_dir = train_cfg["output_dir"]

@dataclass
class DataCollatorWithLabelPadding:
    """Pads input_ids/attention_mask with pad_token_id and labels with -100."""
    tokenizer: Any
    label_pad_token_id: int = -100

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        max_len = max(len(f["input_ids"]) for f in features)
        pad_id = self.tokenizer.pad_token_id
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for f in features:
            pad_len = max_len - len(f["input_ids"])
            batch["input_ids"].append(f["input_ids"] + [pad_id] * pad_len)
            batch["attention_mask"].append(f["attention_mask"] + [0] * pad_len)
            batch["labels"].append(f["labels"] + [self.label_pad_token_id] * pad_len)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=train_cfg.get("num_train_epochs", 3),
    per_device_train_batch_size=train_cfg.get("per_device_train_batch_size", 2),
    per_device_eval_batch_size=train_cfg.get("per_device_eval_batch_size", 2),
    gradient_accumulation_steps=train_cfg.get("gradient_accumulation_steps", 8),
    learning_rate=train_cfg.get("learning_rate", 2e-4),
    warmup_steps=train_cfg.get("warmup_steps", 100),
    logging_steps=train_cfg.get("logging_steps", 10),
    eval_steps=train_cfg.get("eval_steps", 50),
    save_steps=train_cfg.get("save_steps", 100),
    optim=train_cfg.get("optim", "adamw_torch"),
    lr_scheduler_type=train_cfg.get("lr_scheduler_type", "linear"),
    weight_decay=train_cfg.get("weight_decay", 0.01),
    fp16=train_cfg.get("fp16", True),
    bf16=train_cfg.get("bf16", False),
    save_total_limit=train_cfg.get("save_total_limit", 3),
    load_best_model_at_end=train_cfg.get("load_best_model_at_end", True),
    eval_strategy=train_cfg.get("evaluation_strategy", "steps"),
    save_strategy=train_cfg.get("save_strategy", "steps"),
    logging_dir=train_cfg.get("logging_dir", f"{output_dir}/logs"),
    report_to=train_cfg.get("report_to", "none"),
    seed=train_cfg.get("seed", 42),
    gradient_checkpointing=train_cfg.get("gradient_checkpointing", True),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithLabelPadding(tokenizer=tokenizer),
)
print(f"Trainer ready — output: {output_dir}, epochs: {training_args.num_train_epochs}, "
      f"batch: {training_args.per_device_train_batch_size}, lr: {training_args.learning_rate}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Trainer ready — output: ats_phi_lora, epochs: 3, batch: 2, lr: 0.0002


In [19]:
# Run fine-tuning
print("Starting fine-tuning...")
print("=" * 50)

train_result = trainer.train()

print("\n" + "=" * 50)
print("Training complete!")
for key, value in train_result.metrics.items():
    print(f"  {key}: {value}")

Starting fine-tuning...


Step,Training Loss,Validation Loss



Training complete!
  train_runtime: 1013.6967
  train_samples_per_second: 0.533
  train_steps_per_second: 0.036
  total_flos: 1.1802335713062912e+16
  train_loss: 0.0
  epoch: 3.0


In [20]:
# Save LoRA adapter and run final evaluation
print(f"Saving LoRA adapter to: {output_dir}")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("\nSaved files:")
for f in sorted(Path(output_dir).glob("*")):
    print(f"  {f.name}: {f.stat().st_size / 1024:.1f} KB")

print("\nRunning final evaluation...")
eval_results = trainer.evaluate()
for key, value in eval_results.items():
    print(f"  {key}: {value}")

Saving LoRA adapter to: ats_phi_lora

Saved files:
  README.md: 5.1 KB
  adapter_config.json: 1.0 KB
  adapter_model.safetensors: 12296.3 KB
  chat_template.jinja: 0.4 KB
  checkpoint-36: 4.0 KB
  tokenizer.json: 3535.9 KB
  tokenizer_config.json: 0.4 KB

Running final evaluation...


  eval_loss: nan
  eval_runtime: 13.4306
  eval_samples_per_second: 1.489
  eval_steps_per_second: 0.745
  epoch: 3.0


---
## 5. Inference Testing
The trained model is already in memory — switching to eval mode for inference.

In [21]:
import re

model.eval()

def format_inference_prompt(instruction, resume_text, job_description):
    input_text = f"RESUME:\n{resume_text}\n\nJOB DESCRIPTION:\n{job_description}"
    return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

def repair_json(text):
    """Try to fix common JSON issues: truncation, trailing commas, etc."""
    text = text.strip()
    # Remove any text before the first {
    idx = text.find("{")
    if idx == -1:
        return None
    text = text[idx:]
    # Remove trailing commas before } or ]
    text = re.sub(r',\s*([}\]])', r'\1', text)
    # If truncated, try closing open braces/brackets
    open_braces = text.count("{") - text.count("}")
    open_brackets = text.count("[") - text.count("]")
    if open_braces > 0 or open_brackets > 0:
        # Remove any trailing partial key/value (incomplete string)
        text = re.sub(r',?\s*"[^"]*$', '', text)
        text = text.rstrip(', \n\t')
        text += "]" * max(open_brackets, 0)
        text += "}" * max(open_braces, 0)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def extract_json(text):
    """Extract JSON from model output with repair fallback."""
    # Try direct parse first
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # Use non-greedy regex to find the first complete JSON object
    json_match = re.search(r'\{[\s\S]*?\}(?=[^{}]*$)', text)
    if not json_match:
        # Fallback: greedy match (handles nested objects)
        json_match = re.search(r'\{[\s\S]*\}', text)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass
    # Last resort: try to repair truncated/malformed JSON
    return repair_json(text)

def validate_ats_output(output):
    required = {"ats_score", "score_breakdown", "matched_skills",
                "missing_skills", "weak_bullets", "formatting_issues", "overall_feedback"}
    if not isinstance(output, dict) or not required.issubset(output.keys()):
        return False
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        return False
    return True

def generate_ats_eval(resume_text, job_description, max_new_tokens=2048, temperature=0.0):
    instruction = (
        "Evaluate the following resume against the job description and provide a detailed "
        "ATS (Applicant Tracking System) compliance analysis. Return a structured JSON evaluation "
        "including ATS score, score breakdown, matched skills, missing skills, weak bullet analysis "
        "with improvements, formatting issues, and overall feedback."
    )
    prompt = format_inference_prompt(instruction, resume_text, job_description)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    result = extract_json(generated_text)
    if result and validate_ats_output(result):
        result["valid_json"] = True
    else:
        result = {"raw_output": generated_text, "valid_json": False}
    return result

print("Inference helpers ready")


Inference helpers ready


In [22]:
# Test Sample 1
sample_resume = """John Smith
Software Engineer | john.smith@email.com | (555) 123-4567

EXPERIENCE
Software Engineer, TechCorp Inc. - Jan 2021 - Present
- Developed RESTful APIs using Python and Flask serving 10K daily users
- Managed PostgreSQL databases with 50+ tables and optimized query performance
- Collaborated with cross-functional teams to deliver 3 major product releases
- Responsible for maintaining CI/CD pipelines using Jenkins and Docker

Junior Developer, StartupXYZ - Jun 2019 - Dec 2020
- Built frontend components using React and TypeScript
- Helped with bug fixes and code reviews
- Participated in daily standup meetings and sprint planning

EDUCATION
B.S. Computer Science, State University - 2019

SKILLS
Python, JavaScript, React, Flask, PostgreSQL, Docker, Git, AWS"""

sample_jd = """Senior Software Engineer
TechGlobal Inc.

Requirements:
- 5+ years of experience in software development
- Strong proficiency in Python and Java
- Experience with RESTful API design and microservices architecture
- Proficiency with SQL databases (PostgreSQL preferred)
- Experience with cloud services (AWS or GCP)
- Familiarity with Docker and Kubernetes
- Strong understanding of CI/CD pipelines

Responsibilities:
- Design and implement scalable backend services
- Write clean, maintainable, and well-tested code
- Participate in code reviews and technical design discussions
- Mentor junior engineers"""

print("Running inference on sample 1...")
result = generate_ats_eval(sample_resume, sample_jd)
print(json.dumps(result, indent=2))

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Running inference on sample 1...
{
  "raw_output": "```json\n{\n  \"ATS_Score\": 75,\n  \"Score_Breakdown\": {\n    \"Experience\": 80,\n    \"Education\": 70,\n    \"Skills\": 90,\n    \"Responsibilities\": 60\n  },\n  \"Matched_Skills\": [\n    \"Python\",\n    \"JavaScript\",\n    \"React\",\n    \"Flask\",\n    \"PostgreSQL\",\n    \"Docker\",\n    \"Git\",\n    \"AWS\"\n  ],\n  \"Missing_Skills\": [\n    \"Java\",\n    \"Kubernetes\"\n  ],\n  \"Weak_Bullet_Analysis\": {\n    \"Strengths\": [\n      \"Experience with RESTful API design and microservices architecture\",\n      \"Proficiency with SQL databases (PostgreSQL preferred)\",\n      \"Experience with cloud services (AWS or GCP)\"\n    ],\n    \"Weaknesses\": [\n      \"Lack of experience with Java\",\n      \"No mention of CI/CD pipelines experience\",\n      \"No experience with Kubernetes\"\n    ],\n    \"Improvements\": [\n      \"Include Java proficiency in resume\",\n      \"Highlight CI/CD pipeline experience\",\n    

In [23]:
# Validate sample 1 output
if result.get("valid_json"):
    print(f"JSON Validation: PASSED | ATS Score: {result['ats_score']}/100")
    print(f"Matched Skills ({len(result['matched_skills'])}): {result['matched_skills']}")
    print(f"Missing Skills ({len(result['missing_skills'])}): {result['missing_skills']}")
    print(f"Weak Bullets: {len(result['weak_bullets'])} | Formatting Issues: {len(result['formatting_issues'])}")
    print(f"Feedback: {result['overall_feedback'][:200]}...")
else:
    print("JSON Validation: FAILED")
    print(f"Raw: {result.get('raw_output', 'N/A')[:500]}")

JSON Validation: FAILED
Raw: ```json
{
  "ATS_Score": 75,
  "Score_Breakdown": {
    "Experience": 80,
    "Education": 70,
    "Skills": 90,
    "Responsibilities": 60
  },
  "Matched_Skills": [
    "Python",
    "JavaScript",
    "React",
    "Flask",
    "PostgreSQL",
    "Docker",
    "Git",
    "AWS"
  ],
  "Missing_Skills": [
    "Java",
    "Kubernetes"
  ],
  "Weak_Bullet_Analysis": {
    "Strengths": [
      "Experience with RESTful API design and microservices architecture",
      "Proficiency with SQL databases (


In [24]:
# Test Sample 2
sample_resume_2 = """Sarah Chen
Data Scientist | sarah.chen@email.com

EXPERIENCE
Senior Data Scientist, DataDriven Corp - Mar 2022 - Present
- Built machine learning models achieving 92% accuracy on customer churn prediction
- Designed A/B testing framework that increased conversion rates by 15%
- Processed and analyzed datasets containing 10M+ records using PySpark

Data Analyst, Analytics Co - Aug 2020 - Feb 2022
- Created dashboards using Tableau
- Responsible for data cleaning tasks
- Assisted in building predictive models

EDUCATION
M.S. Statistics, Top University - 2020

SKILLS
Python, R, SQL, TensorFlow, PyTorch, Tableau, PySpark, scikit-learn"""

sample_jd_2 = """Machine Learning Engineer
AIVentures Inc.

Requirements:
- 3+ years of ML engineering experience
- Strong Python skills
- Experience with PyTorch or TensorFlow
- Experience deploying ML models to production
- Knowledge of MLOps (MLflow, Kubeflow)
- Experience with NLP or Computer Vision

Responsibilities:
- Design and implement ML pipelines
- Deploy and monitor models in production
- Optimize model performance"""

print("Running inference on sample 2...")
result_2 = generate_ats_eval(sample_resume_2, sample_jd_2)
print(json.dumps(result_2, indent=2))
if result_2.get('valid_json'):
    print(f"\nATS Score: {result_2['ats_score']}/100")

Running inference on sample 2...


KeyboardInterrupt: 

In [ ]:
# Determinism test
print("Testing output consistency (temperature=0.0)...")
result_det1 = generate_ats_eval(sample_resume, sample_jd, temperature=0.0)
result_det2 = generate_ats_eval(sample_resume, sample_jd, temperature=0.0)

if result_det1.get("valid_json") and result_det2.get("valid_json"):
    score_match = result_det1["ats_score"] == result_det2["ats_score"]
    skills_match = result_det1["matched_skills"] == result_det2["matched_skills"]
    print(f"Score match: {score_match} ({result_det1['ats_score']} vs {result_det2['ats_score']})")
    print(f"Skills match: {skills_match}")
    print(f"Deterministic: {'YES' if score_match and skills_match else 'PARTIAL'}")
else:
    print("Cannot test determinism - invalid JSON output")

---
## 6. Evaluation Metrics
Batch inference on the full validation set.

In [ ]:
from tqdm import tqdm

# Load validation data
with open(train_cfg["validation_dataset"], "r", encoding="utf-8") as f:
    eval_val_data = json.load(f)
print(f"Validation samples: {len(eval_val_data)}")

def generate_single(sample, max_new_tokens=2048):
    prompt = f"### Instruction:\n{sample['instruction']}\n\n### Input:\n{sample['input']}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


In [ ]:
# Run batch inference
print(f"Running inference on {len(eval_val_data)} validation samples...\n")

results = []
json_valid_count = 0
ats_valid_count = 0
predicted_scores = []
ground_truth_scores = []
predicted_skills = []
gt_skills = []

for i, sample in enumerate(tqdm(eval_val_data)):
    raw_output = generate_single(sample)
    parsed = extract_json(raw_output)
    gt = json.loads(sample["output"])

    is_valid_json = parsed is not None
    is_valid_ats = is_valid_json and validate_ats_output(parsed)

    if is_valid_json:
        json_valid_count += 1
    if is_valid_ats:
        ats_valid_count += 1
        predicted_scores.append(parsed["ats_score"])
        ground_truth_scores.append(gt["ats_score"])
        predicted_skills.append(set(parsed.get("missing_skills", [])))
        gt_skills.append(set(gt.get("missing_skills", [])))

    results.append({"index": i, "valid_json": is_valid_json, "valid_ats": is_valid_ats,
                     "predicted": parsed, "ground_truth": gt})

print("\nInference complete!")

In [ ]:
# Evaluation metrics
total = len(eval_val_data)
json_rate = json_valid_count / total * 100
ats_rate = ats_valid_count / total * 100

print("=" * 50)
print("EVALUATION METRICS")
print("=" * 50)

print(f"\n1. JSON Validity Rate")
print(f"   Valid JSON: {json_valid_count}/{total} ({json_rate:.1f}%)")
print(f"   Valid ATS:  {ats_valid_count}/{total} ({ats_rate:.1f}%)")
print(f"   Target >95%: {'PASSED' if json_rate >= 95 else 'BELOW TARGET'}")

print(f"\n2. ATS Score Distribution")
if predicted_scores:
    print(f"   Predicted — Min: {min(predicted_scores)}, Max: {max(predicted_scores)}, Mean: {sum(predicted_scores)/len(predicted_scores):.1f}")
    print(f"   Ground truth — Min: {min(ground_truth_scores)}, Max: {max(ground_truth_scores)}, Mean: {sum(ground_truth_scores)/len(ground_truth_scores):.1f}")
    diffs = [abs(p - g) for p, g in zip(predicted_scores, ground_truth_scores)]
    mean_diff = sum(diffs) / len(diffs)
    print(f"   Mean absolute difference: {mean_diff:.1f}")

    buckets = Counter((s // 10) * 10 for s in predicted_scores)
    for bucket in sorted(buckets):
        print(f"     {bucket:3d}-{bucket+9:3d}: {'#' * buckets[bucket]} ({buckets[bucket]})")
else:
    print("   No valid predictions to analyze")

print(f"\n3. Missing Skill Accuracy")
if predicted_skills:
    total_overlap = total_gt = total_pred = 0
    for pred, gt in zip(predicted_skills, gt_skills):
        pred_lower = {s.lower() for s in pred}
        gt_lower = {s.lower() for s in gt}
        total_overlap += len(pred_lower & gt_lower)
        total_gt += len(gt_lower)
        total_pred += len(pred_lower)
    recall = total_overlap / total_gt * 100 if total_gt else 0
    precision = total_overlap / total_pred * 100 if total_pred else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    print(f"   Precision: {precision:.1f}% | Recall: {recall:.1f}% | F1: {f1:.1f}%")
else:
    print("   No valid predictions to analyze")

In [ ]:
# Sample predictions comparison
print("\n" + "=" * 50)
print("SAMPLE PREDICTIONS")
print("=" * 50)

for i, r in enumerate(results[:5]):
    gt = r["ground_truth"]
    pred = r["predicted"]
    print(f"\n--- Sample {i+1} --- JSON: {r['valid_json']} | ATS: {r['valid_ats']}")
    if r["valid_ats"] and pred:
        print(f"  GT Score: {gt['ats_score']} | Pred Score: {pred['ats_score']} | Diff: {abs(gt['ats_score'] - pred['ats_score'])}")
        print(f"  GT Missing:   {gt['missing_skills'][:3]}")
        print(f"  Pred Missing: {pred['missing_skills'][:3]}")
    else:
        print(f"  Prediction: {str(pred)[:200] if pred else 'None'}")

In [ ]:
# Final summary
print("\n" + "=" * 50)
print("EVALUATION SUMMARY")
print("=" * 50)
print(f"  Validation samples: {total}")
print(f"  JSON validity: {json_rate:.1f}%")
print(f"  ATS structure: {ats_rate:.1f}%")
if predicted_scores:
    print(f"  Mean score diff: {mean_diff:.1f}")
print(f"\n  Quality targets:")
print(f"    JSON validity >95%:  {'PASS' if json_rate > 95 else 'FAIL'}")
print(f"    Consistent scoring:  {'PASS' if predicted_scores and mean_diff < 20 else 'NEEDS IMPROVEMENT'}")
print(f"    Valid ATS >90%:      {'PASS' if ats_rate > 90 else 'FAIL'}")
print(f"\n  LoRA adapter saved to: {output_dir}")
print("\nPipeline complete!")

# Task
To address the `valid_json` issue where model output is wrapped in a markdown code block, I will update the `extract_json` function in cell `ad2f6f3e` to correctly parse JSON from these blocks. After updating, I will re-execute the inference helper functions in cell `ad2f6f3e`, followed by re-running the inference tests for `sample_resume` and `sample_jd` in cell `555b1e28`, and `sample_resume_2` and `sample_jd_2` in cell `e8485883`. I will validate the output for sample 1 in cell `3de2f756` to confirm that `valid_json` is now `True`. Next, I will re-run the determinism test in cell `09a1b9b9` and then the full batch inference and evaluation cells `dcffaa30`, `52d3a8e6`, `dd747fb9`, `b48a1015`, `36c1bb3a`, and `91a229a5` to re-evaluate the model's performance with the corrected JSON parsing, ensuring all metrics reflect the fix. Finally, I will confirm the `valid_json` issue is resolved and summarize the overall impact on evaluation metrics.

Here's how I'll modify the `extract_json` function in cell `ad2f6f3e`:

```python
import re

model.eval()

def format_inference_prompt(instruction, resume_text, job_description):
    input_text = f"RESUME:\n{resume_text}\n\nJOB DESCRIPTION:\n{job_description}"
    return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

def repair_json(text):
    """Try to fix common JSON issues: truncation, trailing commas, etc."""
    text = text.strip()
    # Remove any text before the first {
    idx = text.find("{")
    if idx == -1:
        return None
    text = text[idx:]
    # Remove trailing commas before } or ]
    text = re.sub(r',\s*([}\]])', r'\1', text)
    # If truncated, try closing open braces/brackets
    open_braces = text.count("{") - text.count("}")
    open_brackets = text.count("[") - text.count("]")
    if open_braces > 0 or open_brackets > 0:
        # Remove any trailing partial key/value (incomplete string)
        text = re.sub(r',?\s*"[^"]*$', '', text)
        text = text.rstrip(', \n\t')
        text += "]" * max(open_brackets, 0)
        text += "}" * max(open_braces, 0)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def extract_json(text):
    """Extract JSON from model output with repair fallback, handling markdown code blocks."""
    text = text.strip()

    # 1. Check for markdown code block (```json ... ```)
    if text.startswith("```json") and text.endswith("```"):
        text = text[len("```json"): -len("```")].strip()
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            # If it was a markdown block but still invalid, try to repair it
            # This handles cases where the JSON inside the markdown block is itself malformed
            return repair_json(text)

    # 2. Try direct parse first
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass # Fall through to other methods

    # 3. Use non-greedy regex to find the first complete JSON object
    json_match = re.search(r'\{[\s\S]*?\}(?=[^{}]*$)', text)
    if not json_match:
        # Fallback: greedy match (handles nested objects)
        json_match = re.search(r'\{[\s\S]*\}', text)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass

    # 4. Last resort: try to repair truncated/malformed JSON
    return repair_json(text)

def validate_ats_output(output):
    required = {"ats_score", "score_breakdown", "matched_skills",
                "missing_skills", "weak_bullets", "formatting_issues", "overall_feedback"}
    if not isinstance(output, dict) or not required.issubset(output.keys()):
        return False
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        return False
    return True

def generate_ats_eval(resume_text, job_description, max_new_tokens=2048, temperature=0.0):
    instruction = (
        "Evaluate the following resume against the job description and provide a detailed "
        "ATS (Applicant Tracking System) compliance analysis. Return a structured JSON evaluation "
        "including ATS score, score breakdown, matched skills, missing skills, weak bullet analysis "
        "with improvements, formatting issues, and overall feedback."
    )
    prompt = format_inference_prompt(instruction, resume_text, job_description)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    result = extract_json(generated_text)
    if result and validate_ats_output(result):
        result["valid_json"] = True
    else:
        result = {"raw_output": generated_text, "valid_json": False}
    return result

print("Inference helpers ready")
```

## Update extract_json function

### Subtask:
Modify the `extract_json` function in cell `ad2f6f3e` to robustly handle the markdown code block (```json ... ```) that the model output often contains, ensuring that only the raw JSON string is passed to `json.loads`.


**Reasoning**:
The subtask requires modifying the `extract_json` function to handle markdown code blocks. I will update the existing `extract_json` function in cell `ad2f6f3e` to include logic for extracting JSON from ````json ... ````` blocks.



In [25]:
import re

model.eval()

def format_inference_prompt(instruction, resume_text, job_description):
    input_text = f"RESUME:\n{resume_text}\n\nJOB DESCRIPTION:\n{job_description}"
    return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

def repair_json(text):
    """Try to fix common JSON issues: truncation, trailing commas, etc."""
    text = text.strip()
    # Remove any text before the first {
    idx = text.find("{")
    if idx == -1:
        return None
    text = text[idx:]
    # Remove trailing commas before } or ]
    text = re.sub(r',\s*([}\]])', r'\1', text)
    # If truncated, try closing open braces/brackets
    open_braces = text.count("{") - text.count("}")
    open_brackets = text.count("[") - text.count("]")
    if open_braces > 0 or open_brackets > 0:
        # Remove any trailing partial key/value (incomplete string)
        text = re.sub(r',?\s*"[^"]*$', '', text)
        text = text.rstrip(', \n\t')
        text += "]" * max(open_brackets, 0)
        text += "}" * max(open_braces, 0)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def extract_json(text):
    """Extract JSON from model output with repair fallback, handling markdown code blocks."""
    text = text.strip()

    # 1. Try to extract from markdown code block ```json ... ```
    match_md = re.search(r'```json\s*([\s\S]*?)\s*```', text)
    if match_md:
        json_str = match_md.group(1).strip()
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            pass # Fallback to other methods if markdown block is malformed

    # 2. Try direct parse first (if not in markdown or markdown parse failed)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # 3. Use non-greedy regex to find the first complete JSON object
    json_match = re.search(r'\{[\s\S]*?\}(?=[^{}]*$)', text)
    if not json_match:
        # Fallback: greedy match (handles nested objects)
        json_match = re.search(r'\{[\s\S]*\}', text)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass

    # 4. Last resort: try to repair truncated/malformed JSON
    return repair_json(text)

def validate_ats_output(output):
    required = {"ats_score", "score_breakdown", "matched_skills",
                "missing_skills", "weak_bullets", "formatting_issues", "overall_feedback"}
    if not isinstance(output, dict) or not required.issubset(output.keys()):
        return False
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        return False
    return True

def generate_ats_eval(resume_text, job_description, max_new_tokens=2048, temperature=0.0):
    instruction = (
        "Evaluate the following resume against the job description and provide a detailed "
        "ATS (Applicant Tracking System) compliance analysis. Return a structured JSON evaluation "
        "including ATS score, score breakdown, matched skills, missing skills, weak bullet analysis "
        "with improvements, formatting issues, and overall feedback."
    )
    prompt = format_inference_prompt(instruction, resume_text, job_description)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    result = extract_json(generated_text)
    if result and validate_ats_output(result):
        result["valid_json"] = True
    else:
        result = {"raw_output": generated_text, "valid_json": False}
    return result

print("Inference helpers ready")

Inference helpers ready


**Reasoning**:
Now that the `extract_json` function has been updated to handle markdown code blocks, I will re-run the inference test for `sample_resume` and `sample_jd` in cell `555b1e28` to observe the effect of the change on the `valid_json` flag.



In [26]:
print("Running inference on sample 1...")
result = generate_ats_eval(sample_resume, sample_jd)
print(json.dumps(result, indent=2))

Running inference on sample 1...
{
  "raw_output": "```json\n{\n  \"ATS_Score\": 75,\n  \"Score_Breakdown\": {\n    \"Experience\": 80,\n    \"Education\": 70,\n    \"Skills\": 90,\n    \"Responsibilities\": 60\n  },\n  \"Matched_Skills\": [\n    \"Python\",\n    \"JavaScript\",\n    \"React\",\n    \"Flask\",\n    \"PostgreSQL\",\n    \"Docker\",\n    \"Git\",\n    \"AWS\"\n  ],\n  \"Missing_Skills\": [\n    \"Java\",\n    \"Kubernetes\"\n  ],\n  \"Weak_Bullet_Analysis\": {\n    \"Strengths\": [\n      \"Experience with RESTful API design and microservices architecture\",\n      \"Proficiency with SQL databases (PostgreSQL preferred)\",\n      \"Experience with cloud services (AWS or GCP)\"\n    ],\n    \"Weaknesses\": [\n      \"Lack of experience with Java\",\n      \"No mention of CI/CD pipelines experience\",\n      \"No experience with Kubernetes\"\n    ],\n    \"Improvements\": [\n      \"Include Java proficiency in resume\",\n      \"Highlight CI/CD pipeline experience\",\n    

**Reasoning**:
The previous inference test indicated that `valid_json` is still `False` even after the `extract_json` function was updated. This is likely due to a mismatch in key casing (e.g., 'ATS_Score' vs 'ats_score') between the model's output and the `validate_ats_output` function's expectations. To fix this, I will modify the `extract_json` function to normalize all keys to snake_case after parsing the JSON. This will ensure consistency and allow `validate_ats_output` to correctly validate the structure.



In [27]:
import json
import re
import torch

model.eval()

def format_inference_prompt(instruction, resume_text, job_description):
    input_text = f"RESUME:\n{resume_text}\n\nJOB DESCRIPTION:\n{job_description}"
    return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

def repair_json(text):
    """Try to fix common JSON issues: truncation, trailing commas, etc."""
    text = text.strip()
    # Remove any text before the first {
    idx = text.find("{")
    if idx == -1:
        return None
    text = text[idx:]
    # Remove trailing commas before } or ]
    text = re.sub(r',\s*([}\]])', r'\1', text)
    # If truncated, try closing open braces/brackets
    open_braces = text.count("{") - text.count("}")
    open_brackets = text.count("[") - text.count("]")
    if open_braces > 0 or open_brackets > 0:
        # Remove any trailing partial key/value (incomplete string)
        text = re.sub(r',?\s*"[^"]*$', '', text)
        text = text.rstrip(', \n\t')
        text += "]" * max(open_brackets, 0)
        text += "}" * max(open_braces, 0)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def to_snake_case(name):
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()

def _normalize_json_keys(obj):
    if isinstance(obj, dict):
        return {to_snake_case(k): _normalize_json_keys(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [_normalize_json_keys(elem) for elem in obj]
    else:
        return obj

def extract_json(text):
    """Extract JSON from model output with repair fallback, handling markdown code blocks and normalizing keys."""
    text = text.strip()
    parsed_json = None

    # 1. Try to extract from markdown code block ```json ... ```
    match_md = re.search(r'```json\s*([\s\S]*?)\s*```', text)
    if match_md:
        json_str = match_md.group(1).strip()
        try:
            parsed_json = json.loads(json_str)
        except json.JSONDecodeError:
            pass # Fallback to other methods if markdown block is malformed

    # 2. Try direct parse if not yet parsed (if not in markdown or markdown parse failed)
    if parsed_json is None:
        try:
            parsed_json = json.loads(text)
        except json.JSONDecodeError:
            pass

    # 3. Use non-greedy regex to find the first complete JSON object if not yet parsed
    if parsed_json is None:
        json_match = re.search(r'\{[\s\S]*?\}(?=[^{}]*$)', text)
        if not json_match:
            # Fallback: greedy match (handles nested objects)
            json_match = re.search(r'\{[\s\S]*\}', text)
        if json_match:
            try:
                parsed_json = json.loads(json_match.group())
            except json.JSONDecodeError:
                pass

    # 4. Last resort: try to repair truncated/malformed JSON if not yet parsed
    if parsed_json is None:
        parsed_json = repair_json(text)

    # Normalize keys if JSON was successfully parsed
    if parsed_json is not None:
        return _normalize_json_keys(parsed_json)
    return None

def validate_ats_output(output):
    required = {"ats_score", "score_breakdown", "matched_skills",
                "missing_skills", "weak_bullets", "formatting_issues", "overall_feedback"}
    if not isinstance(output, dict) or not required.issubset(output.keys()):
        return False
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        return False
    return True

def generate_ats_eval(resume_text, job_description, max_new_tokens=2048, temperature=0.0):
    instruction = (
        "Evaluate the following resume against the job description and provide a detailed "
        "ATS (Applicant Tracking System) compliance analysis. Return a structured JSON evaluation "
        "including ATS score, score breakdown, matched skills, missing skills, weak bullet analysis "
        "with improvements, formatting issues, and overall feedback."
    )
    prompt = format_inference_prompt(instruction, resume_text, job_description)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    result = extract_json(generated_text)
    if result and validate_ats_output(result):
        result["valid_json"] = True
    else:
        result = {"raw_output": generated_text, "valid_json": False}
    return result

print("Inference helpers ready")

Inference helpers ready


**Reasoning**:
Now that the `extract_json` function has been updated to include key normalization, I will re-run the inference test for `sample_resume` and `sample_jd` in cell `555b1e28` to confirm that the `valid_json` flag is now `True`.



In [28]:
print("Running inference on sample 1...")
result = generate_ats_eval(sample_resume, sample_jd)
print(json.dumps(result, indent=2))

Running inference on sample 1...
{
  "raw_output": "```json\n{\n  \"ATS_Score\": 75,\n  \"Score_Breakdown\": {\n    \"Experience\": 80,\n    \"Education\": 70,\n    \"Skills\": 90,\n    \"Responsibilities\": 60\n  },\n  \"Matched_Skills\": [\n    \"Python\",\n    \"JavaScript\",\n    \"React\",\n    \"Flask\",\n    \"PostgreSQL\",\n    \"Docker\",\n    \"Git\",\n    \"AWS\"\n  ],\n  \"Missing_Skills\": [\n    \"Java\",\n    \"Kubernetes\"\n  ],\n  \"Weak_Bullet_Analysis\": {\n    \"Strengths\": [\n      \"Experience with RESTful API design and microservices architecture\",\n      \"Proficiency with SQL databases (PostgreSQL preferred)\",\n      \"Experience with cloud services (AWS or GCP)\"\n    ],\n    \"Weaknesses\": [\n      \"Lack of experience with Java\",\n      \"No mention of CI/CD pipelines experience\",\n      \"No experience with Kubernetes\"\n    ],\n    \"Improvements\": [\n      \"Include Java proficiency in resume\",\n      \"Highlight CI/CD pipeline experience\",\n    

**Reasoning**:
The `valid_json` flag is still `False`, indicating an issue with `extract_json` or `validate_ats_output`. To debug this, I will add print statements within the `extract_json` function to inspect the `json_str` after markdown extraction, the `parsed_json` before normalization, and the `normalized_json` returned. I will also add print statements in `generate_ats_eval` to show the raw `generated_text` and the result from `extract_json`.



In [29]:
import json
import re
import torch

model.eval()

def format_inference_prompt(instruction, resume_text, job_description):
    input_text = f"RESUME:\n{resume_text}\n\nJOB DESCRIPTION:\n{job_description}"
    return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

def repair_json(text):
    """Try to fix common JSON issues: truncation, trailing commas, etc."""
    text = text.strip()
    # Remove any text before the first {
    idx = text.find("{")
    if idx == -1:
        return None
    text = text[idx:]
    # Remove trailing commas before } or ]
    text = re.sub(r',\s*([}\]])', r'\1', text)
    # If truncated, try closing open braces/brackets
    open_braces = text.count("{") - text.count("}")
    open_brackets = text.count("[") - text.count("]")
    if open_braces > 0 or open_brackets > 0:
        # Remove any trailing partial key/value (incomplete string)
        text = re.sub(r',?\s*"[^"]*$", '', text)
        text = text.rstrip(', \n\t')
        text += "]" * max(open_brackets, 0)
        text += "}" * max(open_braces, 0)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def to_snake_case(name):
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()

def _normalize_json_keys(obj):
    if isinstance(obj, dict):
        return {to_snake_case(k): _normalize_json_keys(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [_normalize_json_keys(elem) for elem in obj]
    else:
        return obj

def extract_json(text):
    """Extract JSON from model output with repair fallback, handling markdown code blocks and normalizing keys."""
    text = text.strip()
    parsed_json = None

    print(f"[DEBUG extract_json] Initial text (first 200 chars): {text[:200]}...")

    # 1. Try to extract from markdown code block ```json ... ```
    match_md = re.search(r'```json\s*([\s\S]*?)\s*```', text)
    if match_md:
        json_str = match_md.group(1).strip()
        print(f"[DEBUG extract_json] Markdown JSON string extracted (first 200 chars): {json_str[:200]}...")
        try:
            parsed_json = json.loads(json_str)
            print("[DEBUG extract_json] Parsed from markdown successfully.")
        except json.JSONDecodeError as e:
            print(f"[DEBUG extract_json] JSONDecodeError in markdown block: {e}")
            pass # Fallback to other methods if markdown block is malformed

    # 2. Try direct parse if not yet parsed (if not in markdown or markdown parse failed)
    if parsed_json is None:
        print("[DEBUG extract_json] Trying direct parse.")
        try:
            parsed_json = json.loads(text)
            print("[DEBUG extract_json] Direct parse successful.")
        except json.JSONDecodeError:
            print("[DEBUG extract_json] Direct parse failed.")
            pass

    # 3. Use non-greedy regex to find the first complete JSON object if not yet parsed
    if parsed_json is None:
        print("[DEBUG extract_json] Trying regex search for JSON.")
        json_match = re.search(r'\{[\s\S]*?\}(?=[^{}]*$)', text)
        if not json_match:
            # Fallback: greedy match (handles nested objects)
            json_match = re.search(r'\{[\s\S]*\}', text)
        if json_match:
            try:
                parsed_json = json.loads(json_match.group())
                print("[DEBUG extract_json] Regex JSON parse successful.")
            except json.JSONDecodeError:
                print("[DEBUG extract_json] Regex JSON parse failed.")
                pass

    # 4. Last resort: try to repair truncated/malformed JSON if not yet parsed
    if parsed_json is None:
        print("[DEBUG extract_json] Trying repair_json.")
        parsed_json = repair_json(text)
        if parsed_json:
            print("[DEBUG extract_json] repair_json successful.")
        else:
            print("[DEBUG extract_json] repair_json failed.")

    # Normalize keys if JSON was successfully parsed
    if parsed_json is not None:
        normalized_json = _normalize_json_keys(parsed_json)
        print(f"[DEBUG extract_json] Normalized JSON (first key): {list(normalized_json.keys())[0] if normalized_json else 'None'}")
        return normalized_json

    print("[DEBUG extract_json] No valid JSON could be extracted.")
    return None

def validate_ats_output(output):
    required = {"ats_score", "score_breakdown", "matched_skills",
                "missing_skills", "weak_bullets", "formatting_issues", "overall_feedback"}
    if not isinstance(output, dict) or not required.issubset(output.keys()):
        print(f"[DEBUG validate_ats_output] Failed: Not a dict or missing keys. Type: {type(output)}, Keys: {output.keys() if isinstance(output, dict) else 'N/A'}")
        return False
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        print(f"[DEBUG validate_ats_output] Failed: Invalid ats_score. Score: {output['ats_score']}")
        return False
    print("[DEBUG validate_ats_output] Passed validation.")
    return True

def generate_ats_eval(resume_text, job_description, max_new_tokens=2048, temperature=0.0):
    instruction = (
        "Evaluate the following resume against the job description and provide a detailed "
        "ATS (Applicant Tracking System) compliance analysis. Return a structured JSON evaluation "
        "including ATS score, score breakdown, matched skills, missing skills, weak bullet analysis "
        "with improvements, formatting issues, and overall feedback."
    )
    prompt = format_inference_prompt(instruction, resume_text, job_description)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    print(f"[DEBUG generate_ats_eval] Raw generated text (first 500 chars): {generated_text[:500]}...")

    result = extract_json(generated_text)

    print(f"[DEBUG generate_ats_eval] Result from extract_json: {json.dumps(result, indent=2) if result else 'None'}")

    if result and validate_ats_output(result):
        result["valid_json"] = True
    else:
        # Ensure 'raw_output' is explicitly set when valid_json is False
        if not isinstance(result, dict) or "raw_output" not in result:
            result = {"raw_output": generated_text, "valid_json": False}
        else:
            result["valid_json"] = False # In case result is a dict but validation failed

    return result

print("Inference helpers ready")

SyntaxError: unterminated string literal (detected at line 26) (1011960527.py, line 26)

**Reasoning**:
The previous code block failed due to a `SyntaxError: unterminated string literal` in the `repair_json` function. This was caused by an unescaped double quote within a regex pattern. I will fix this by correctly escaping the double quote to `\"`.



In [30]:
import json
import re
import torch

model.eval()

def format_inference_prompt(instruction, resume_text, job_description):
    input_text = f"RESUME:\n{resume_text}\n\nJOB DESCRIPTION:\n{job_description}"
    return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

def repair_json(text):
    """Try to fix common JSON issues: truncation, trailing commas, etc."""
    text = text.strip()
    # Remove any text before the first {
    idx = text.find("{")
    if idx == -1:
        return None
    text = text[idx:]
    # Remove trailing commas before } or ]
    text = re.sub(r',\s*([}\]])', r'\1', text)
    # If truncated, try closing open braces/brackets
    open_braces = text.count("{") - text.count("}")
    open_brackets = text.count("[") - text.count("]")
    if open_braces > 0 or open_brackets > 0:
        # Remove any trailing partial key/value (incomplete string)
        text = re.sub(r',?\s*\"[^\"]*$', '', text)
        text = text.rstrip(', \n\t')
        text += "]" * max(open_brackets, 0)
        text += "}" * max(open_braces, 0)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def to_snake_case(name):
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()

def _normalize_json_keys(obj):
    if isinstance(obj, dict):
        return {to_snake_case(k): _normalize_json_keys(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [_normalize_json_keys(elem) for elem in obj]
    else:
        return obj

def extract_json(text):
    """Extract JSON from model output with repair fallback, handling markdown code blocks and normalizing keys."""
    text = text.strip()
    parsed_json = None

    print(f"[DEBUG extract_json] Initial text (first 200 chars): {text[:200]}...")

    # 1. Try to extract from markdown code block ```json ... ```
    match_md = re.search(r'```json\s*([\s\S]*?)\s*```', text)
    if match_md:
        json_str = match_md.group(1).strip()
        print(f"[DEBUG extract_json] Markdown JSON string extracted (first 200 chars): {json_str[:200]}...")
        try:
            parsed_json = json.loads(json_str)
            print("[DEBUG extract_json] Parsed from markdown successfully.")
        except json.JSONDecodeError as e:
            print(f"[DEBUG extract_json] JSONDecodeError in markdown block: {e}")
            pass # Fallback to other methods if markdown block is malformed

    # 2. Try direct parse if not yet parsed (if not in markdown or markdown parse failed)
    if parsed_json is None:
        print("[DEBUG extract_json] Trying direct parse.")
        try:
            parsed_json = json.loads(text)
            print("[DEBUG extract_json] Direct parse successful.")
        except json.JSONDecodeError:
            print("[DEBUG extract_json] Direct parse failed.")
            pass

    # 3. Use non-greedy regex to find the first complete JSON object if not yet parsed
    if parsed_json is None:
        print("[DEBUG extract_json] Trying regex search for JSON.")
        json_match = re.search(r'\{[\s\S]*?\}(?=[^{}]*$)', text)
        if not json_match:
            # Fallback: greedy match (handles nested objects)
            json_match = re.search(r'\{[\s\S]*\}', text)
        if json_match:
            try:
                parsed_json = json.loads(json_match.group())
                print("[DEBUG extract_json] Regex JSON parse successful.")
            except json.JSONDecodeError:
                print("[DEBUG extract_json] Regex JSON parse failed.")
                pass

    # 4. Last resort: try to repair truncated/malformed JSON if not yet parsed
    if parsed_json is None:
        print("[DEBUG extract_json] Trying repair_json.")
        parsed_json = repair_json(text)
        if parsed_json:
            print("[DEBUG extract_json] repair_json successful.")
        else:
            print("[DEBUG extract_json] repair_json failed.")

    # Normalize keys if JSON was successfully parsed
    if parsed_json is not None:
        normalized_json = _normalize_json_keys(parsed_json)
        print(f"[DEBUG extract_json] Normalized JSON (first key): {list(normalized_json.keys())[0] if normalized_json else 'None'}")
        return normalized_json

    print("[DEBUG extract_json] No valid JSON could be extracted.")
    return None

def validate_ats_output(output):
    required = {"ats_score", "score_breakdown", "matched_skills",
                "missing_skills", "weak_bullets", "formatting_issues", "overall_feedback"}
    if not isinstance(output, dict) or not required.issubset(output.keys()):
        print(f"[DEBUG validate_ats_output] Failed: Not a dict or missing keys. Type: {type(output)}, Keys: {output.keys() if isinstance(output, dict) else 'N/A'}")
        return False
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        print(f"[DEBUG validate_ats_output] Failed: Invalid ats_score. Score: {output["ats_score"]}")
        return False
    print("[DEBUG validate_ats_output] Passed validation.")
    return True

def generate_ats_eval(resume_text, job_description, max_new_tokens=2048, temperature=0.0):
    instruction = (
        "Evaluate the following resume against the job description and provide a detailed "
        "ATS (Applicant Tracking System) compliance analysis. Return a structured JSON evaluation "
        "including ATS score, score breakdown, matched skills, missing skills, weak bullet analysis "
        "with improvements, formatting issues, and overall feedback."
    )
    prompt = format_inference_prompt(instruction, resume_text, job_description)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    print(f"[DEBUG generate_ats_eval] Raw generated text (first 500 chars): {generated_text[:500]}...")

    result = extract_json(generated_text)

    print(f"[DEBUG generate_ats_eval] Result from extract_json: {json.dumps(result, indent=2) if result else 'None'}")

    if result and validate_ats_output(result):
        result["valid_json"] = True
    else:
        # Ensure 'raw_output' is explicitly set when valid_json is False
        if not isinstance(result, dict) or "raw_output" not in result:
            result = {"raw_output": generated_text, "valid_json": False}
        else:
            result["valid_json"] = False # In case result is a dict but validation failed

    return result

print("Inference helpers ready")

Inference helpers ready


**Reasoning**:
Now that the `extract_json` function has been updated with debug prints and the `SyntaxError` fixed, I will re-run the inference test for `sample_resume` and `sample_jd` to observe the debug output and the `valid_json` flag.



In [31]:
print("Running inference on sample 1...")
result = generate_ats_eval(sample_resume, sample_jd)
print(json.dumps(result, indent=2))

Running inference on sample 1...
[DEBUG generate_ats_eval] Raw generated text (first 500 chars): ```json
{
  "ATS_Score": 75,
  "Score_Breakdown": {
    "Experience": 80,
    "Education": 70,
    "Skills": 90,
    "Responsibilities": 60
  },
  "Matched_Skills": [
    "Python",
    "JavaScript",
    "React",
    "Flask",
    "PostgreSQL",
    "Docker",
    "Git",
    "AWS"
  ],
  "Missing_Skills": [
    "Java",
    "Kubernetes"
  ],
  "Weak_Bullet_Analysis": {
    "Strengths": [
      "Experience with RESTful API design and microservices architecture",
      "Proficiency with SQL databases (...
[DEBUG extract_json] Initial text (first 200 chars): ```json
{
  "ATS_Score": 75,
  "Score_Breakdown": {
    "Experience": 80,
    "Education": 70,
    "Skills": 90,
    "Responsibilities": 60
  },
  "Matched_Skills": [
    "Python",
    "JavaScript",
 ...
[DEBUG extract_json] Markdown JSON string extracted (first 200 chars): {
  "ATS_Score": 75,
  "Score_Breakdown": {
    "Experience": 80,
    "

**Reasoning**:
The debug output shows that keys like 'ats_score' are being normalized to 'ats__score' (double underscore) instead of 'ats_score' (single underscore), which causes `validate_ats_output` to fail because the required keys are not found. I need to correct the `to_snake_case` function to produce proper snake_case (single underscore) from PascalCase or existing snake_case keys. I will update the `to_snake_case` function in cell `ad2f6f3e` and re-run the inference helpers.



In [32]:
import json
import re
import torch

model.eval()

def format_inference_prompt(instruction, resume_text, job_description):
    input_text = f"RESUME:\n{resume_text}\n\nJOB DESCRIPTION:\n{job_description}"
    return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

def repair_json(text):
    """Try to fix common JSON issues: truncation, trailing commas, etc."""
    text = text.strip()
    # Remove any text before the first {
    idx = text.find("{")
    if idx == -1:
        return None
    text = text[idx:]
    # Remove trailing commas before } or ]
    text = re.sub(r',\s*([}\]])', r'\1', text)
    # If truncated, try closing open braces/brackets
    open_braces = text.count("{") - text.count("}")
    open_brackets = text.count("[") - text.count("]")
    if open_braces > 0 or open_brackets > 0:
        # Remove any trailing partial key/value (incomplete string)
        text = re.sub(r',?\s*\"[^\"]*$", '', text)
        text = text.rstrip(', \n\t')
        text += "]" * max(open_brackets, 0)
        text += "}" * max(open_braces, 0)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def to_snake_case(name):
    # Convert PascalCase/CamelCase to snake_case
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    s2 = re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()
    # Replace double underscores with single ones, if any were created by existing snake_case keys like ATS_Score
    return re.sub(r'__+', '_', s2)

def _normalize_json_keys(obj):
    if isinstance(obj, dict):
        return {to_snake_case(k): _normalize_json_keys(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [_normalize_json_keys(elem) for elem in obj]
    else:
        return obj

def extract_json(text):
    """Extract JSON from model output with repair fallback, handling markdown code blocks and normalizing keys."""
    text = text.strip()
    parsed_json = None

    # 1. Try to extract from markdown code block ```json ... ```
    match_md = re.search(r'```json\s*([\s\S]*?)\s*```', text)
    if match_md:
        json_str = match_md.group(1).strip()
        try:
            parsed_json = json.loads(json_str)
        except json.JSONDecodeError:
            pass # Fallback to other methods if markdown block is malformed

    # 2. Try direct parse if not yet parsed (if not in markdown or markdown parse failed)
    if parsed_json is None:
        try:
            parsed_json = json.loads(text)
        except json.JSONDecodeError:
            pass

    # 3. Use non-greedy regex to find the first complete JSON object if not yet parsed
    if parsed_json is None:
        json_match = re.search(r'\{[\s\S]*?\}(?=[^{}]*$)', text)
        if not json_match:
            # Fallback: greedy match (handles nested objects)
            json_match = re.search(r'\{[\s\S]*\}', text)
        if json_match:
            try:
                parsed_json = json.loads(json_match.group())
            except json.JSONDecodeError:
                pass

    # 4. Last resort: try to repair truncated/malformed JSON if not yet parsed
    if parsed_json is None:
        parsed_json = repair_json(text)

    # Normalize keys if JSON was successfully parsed
    if parsed_json is not None:
        return _normalize_json_keys(parsed_json)
    return None

def validate_ats_output(output):
    required = {"ats_score", "score_breakdown", "matched_skills",
                "missing_skills", "weak_bullets", "formatting_issues", "overall_feedback"}
    if not isinstance(output, dict) or not required.issubset(output.keys()):
        return False
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        return False
    return True

def generate_ats_eval(resume_text, job_description, max_new_tokens=2048, temperature=0.0):
    instruction = (
        "Evaluate the following resume against the job description and provide a detailed "
        "ATS (Applicant Tracking System) compliance analysis. Return a structured JSON evaluation "
        "including ATS score, score breakdown, matched skills, missing skills, weak bullet analysis "
        "with improvements, formatting issues, and overall feedback."
    )
    prompt = format_inference_prompt(instruction, resume_text, job_description)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    result = extract_json(generated_text)
    if result and validate_ats_output(result):
        result["valid_json"] = True
    else:
        result = {"raw_output": generated_text, "valid_json": False}
    return result

print("Inference helpers ready")

SyntaxError: unterminated string literal (detected at line 26) (1356890280.py, line 26)

**Reasoning**:
The previous code block failed due to a `SyntaxError: unterminated string literal` in the `repair_json` function, caused by an incorrectly escaped double quote in a regex pattern. Additionally, the `to_snake_case` function was over-inserting underscores (e.g., `ATS_Score` becoming `ats__score`), leading to validation failures. I will fix the `repair_json` regex by correcting the double quote escape and simplify the `to_snake_case` function to only lowercase the keys, as the model's output keys are already in a snake_case-like structure with varying capitalization. I will also remove the debug print statements.



In [33]:
import json
import re
import torch

model.eval()

def format_inference_prompt(instruction, resume_text, job_description):
    input_text = f"RESUME:\n{resume_text}\n\nJOB DESCRIPTION:\n{job_description}"
    return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

def repair_json(text):
    """Try to fix common JSON issues: truncation, trailing commas, etc."""
    text = text.strip()
    # Remove any text before the first {
    idx = text.find("{")
    if idx == -1:
        return None
    text = text[idx:]
    # Remove trailing commas before } or ]
    text = re.sub(r',\s*([}\]])', r'\1', text)
    # If truncated, try closing open braces/brackets
    open_braces = text.count("{") - text.count("}")
    open_brackets = text.count("[") - text.count("]")
    if open_braces > 0 or open_brackets > 0:
        # Remove any trailing partial key/value (incomplete string)
        text = re.sub(r',?\s*"[^"]*$", '', text)
        text = text.rstrip(', \n\t')
        text += "]" * max(open_brackets, 0)
        text += "}" * max(open_braces, 0)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def to_snake_case(name):
    # Model output keys like 'ATS_Score' or 'Score_Breakdown' just need to be lowercased.
    return name.lower()

def _normalize_json_keys(obj):
    if isinstance(obj, dict):
        return {to_snake_case(k): _normalize_json_keys(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [_normalize_json_keys(elem) for elem in obj]
    else:
        return obj

def extract_json(text):
    """Extract JSON from model output with repair fallback, handling markdown code blocks and normalizing keys."""
    text = text.strip()
    parsed_json = None

    # 1. Try to extract from markdown code block ```json ... ```
    match_md = re.search(r'```json\s*([\s\S]*?)\s*```', text)
    if match_md:
        json_str = match_md.group(1).strip()
        try:
            parsed_json = json.loads(json_str)
        except json.JSONDecodeError:
            pass # Fallback to other methods if markdown block is malformed

    # 2. Try direct parse if not yet parsed (if not in markdown or markdown parse failed)
    if parsed_json is None:
        try:
            parsed_json = json.loads(text)
        except json.JSONDecodeError:
            pass

    # 3. Use non-greedy regex to find the first complete JSON object if not yet parsed
    if parsed_json is None:
        json_match = re.search(r'\{[\s\S]*?\}(?=[^{}]*$)', text)
        if not json_match:
            # Fallback: greedy match (handles nested objects)
            json_match = re.search(r'\{[\s\S]*\}', text)
        if json_match:
            try:
                parsed_json = json.loads(json_match.group())
            except json.JSONDecodeError:
                pass

    # 4. Last resort: try to repair truncated/malformed JSON if not yet parsed
    if parsed_json is None:
        parsed_json = repair_json(text)

    # Normalize keys if JSON was successfully parsed
    if parsed_json is not None:
        return _normalize_json_keys(parsed_json)
    return None

def validate_ats_output(output):
    required = {"ats_score", "score_breakdown", "matched_skills",
                "missing_skills", "weak_bullets", "formatting_issues", "overall_feedback"}
    if not isinstance(output, dict) or not required.issubset(output.keys()):
        return False
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        return False
    return True

def generate_ats_eval(resume_text, job_description, max_new_tokens=2048, temperature=0.0):
    instruction = (
        "Evaluate the following resume against the job description and provide a detailed "
        "ATS (Applicant Tracking System) compliance analysis. Return a structured JSON evaluation "
        "including ATS score, score breakdown, matched skills, missing skills, weak bullet analysis "
        "with improvements, formatting issues, and overall feedback."
    )
    prompt = format_inference_prompt(instruction, resume_text, job_description)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    result = extract_json(generated_text)
    if result and validate_ats_output(result):
        result["valid_json"] = True
    else:
        result = {"raw_output": generated_text, "valid_json": False}
    return result

print("Inference helpers ready")

SyntaxError: unterminated string literal (detected at line 26) (3086898254.py, line 26)

**Reasoning**:
The `SyntaxError` in the `repair_json` function was caused by an incorrectly placed double quote in the regex pattern. I will revert the regex pattern to its original, correct form `r',?\s*"[^"]*$'` and simplify the `to_snake_case` function to only lowercase keys, which matches the model's output format more effectively. Additionally, I will remove the debug print statements and revert the `generate_ats_eval` function to its original error handling logic for a cleaner, more robust solution.



In [34]:
import json
import re
import torch

model.eval()

def format_inference_prompt(instruction, resume_text, job_description):
    input_text = f"RESUME:\n{resume_text}\n\nJOB DESCRIPTION:\n{job_description}"
    return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

def repair_json(text):
    """Try to fix common JSON issues: truncation, trailing commas, etc."""
    text = text.strip()
    # Remove any text before the first {
    idx = text.find("{")
    if idx == -1:
        return None
    text = text[idx:]
    # Remove trailing commas before } or ]
    text = re.sub(r',\s*([}\]])', r'\1', text)
    # If truncated, try closing open braces/brackets
    open_braces = text.count("{") - text.count("}")
    open_brackets = text.count("[") - text.count("]")
    if open_braces > 0 or open_brackets > 0:
        # Remove any trailing partial key/value (incomplete string)
        text = re.sub(r',?\s*"[^"]*$', '', text)
        text = text.rstrip(', \n\t')
        text += "]" * max(open_brackets, 0)
        text += "}" * max(open_braces, 0)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def to_snake_case(name):
    # Model output keys like 'ATS_Score' or 'Score_Breakdown' just need to be lowercased.
    return name.lower()

def _normalize_json_keys(obj):
    if isinstance(obj, dict):
        return {to_snake_case(k): _normalize_json_keys(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [_normalize_json_keys(elem) for elem in obj]
    else:
        return obj

def extract_json(text):
    """Extract JSON from model output with repair fallback, handling markdown code blocks and normalizing keys."""
    text = text.strip()
    parsed_json = None

    # 1. Try to extract from markdown code block ```json ... ```
    match_md = re.search(r'```json\s*([\s\S]*?)\s*```', text)
    if match_md:
        json_str = match_md.group(1).strip()
        try:
            parsed_json = json.loads(json_str)
        except json.JSONDecodeError:
            pass # Fallback to other methods if markdown block is malformed

    # 2. Try direct parse if not yet parsed (if not in markdown or markdown parse failed)
    if parsed_json is None:
        try:
            parsed_json = json.loads(text)
        except json.JSONDecodeError:
            pass

    # 3. Use non-greedy regex to find the first complete JSON object if not yet parsed
    if parsed_json is None:
        json_match = re.search(r'\{([\s\S]*?)\}(?=[^{}]*$)', text)
        if not json_match:
            # Fallback: greedy match (handles nested objects)
            json_match = re.search(r'\{[\s\S]*\}', text)
        if json_match:
            try:
                parsed_json = json.loads(json_match.group())
            except json.JSONDecodeError:
                pass

    # 4. Last resort: try to repair truncated/malformed JSON if not yet parsed
    if parsed_json is None:
        parsed_json = repair_json(text)

    # Normalize keys if JSON was successfully parsed
    if parsed_json is not None:
        return _normalize_json_keys(parsed_json)
    return None

def validate_ats_output(output):
    required = {"ats_score", "score_breakdown", "matched_skills",
                "missing_skills", "weak_bullets", "formatting_issues", "overall_feedback"}
    if not isinstance(output, dict) or not required.issubset(output.keys()):
        return False
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        return False
    return True

def generate_ats_eval(resume_text, job_description, max_new_tokens=2048, temperature=0.0):
    instruction = (
        "Evaluate the following resume against the job description and provide a detailed "
        "ATS (Applicant Tracking System) compliance analysis. Return a structured JSON evaluation "
        "including ATS score, score breakdown, matched skills, missing skills, weak bullet analysis "
        "with improvements, formatting issues, and overall feedback."
    )
    prompt = format_inference_prompt(instruction, resume_text, job_description)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    result = extract_json(generated_text)
    if result and validate_ats_output(result):
        result["valid_json"] = True
    else:
        result = {"raw_output": generated_text, "valid_json": False}
    return result

print("Inference helpers ready")

Inference helpers ready


**Reasoning**:
Now that the `extract_json` function has been updated with the corrected regex for `repair_json`, simplified `to_snake_case` function, and removed debug prints, I will re-run the inference test for `sample_resume` and `sample_jd` in cell `555b1e28` to confirm that the `valid_json` flag is now `True`.



In [35]:
print("Running inference on sample 1...")
result = generate_ats_eval(sample_resume, sample_jd)
print(json.dumps(result, indent=2))

Running inference on sample 1...
{
  "raw_output": "```json\n{\n  \"ATS_Score\": 75,\n  \"Score_Breakdown\": {\n    \"Experience\": 80,\n    \"Education\": 70,\n    \"Skills\": 90,\n    \"Responsibilities\": 60\n  },\n  \"Matched_Skills\": [\n    \"Python\",\n    \"JavaScript\",\n    \"React\",\n    \"Flask\",\n    \"PostgreSQL\",\n    \"Docker\",\n    \"Git\",\n    \"AWS\"\n  ],\n  \"Missing_Skills\": [\n    \"Java\",\n    \"Kubernetes\"\n  ],\n  \"Weak_Bullet_Analysis\": {\n    \"Strengths\": [\n      \"Experience with RESTful API design and microservices architecture\",\n      \"Proficiency with SQL databases (PostgreSQL preferred)\",\n      \"Experience with cloud services (AWS or GCP)\"\n    ],\n    \"Weaknesses\": [\n      \"Lack of experience with Java\",\n      \"No mention of CI/CD pipelines experience\",\n      \"No experience with Kubernetes\"\n    ],\n    \"Improvements\": [\n      \"Include Java proficiency in resume\",\n      \"Highlight CI/CD pipeline experience\",\n    

**Reasoning**:
The previous inference test indicated that `valid_json` is still `False`. To debug the `extract_json` function and understand why it's failing to parse the JSON or why the `validate_ats_output` function is returning `False`, I will re-introduce debug print statements into the `extract_json` and `validate_ats_output` functions in cell `941145d1`. This will allow me to inspect the extracted JSON string, the parsed JSON before and after normalization, and the reasons for validation failure.



In [36]:
import json
import re
import torch

model.eval()

def format_inference_prompt(instruction, resume_text, job_description):
    input_text = f"RESUME:\n{resume_text}\n\nJOB DESCRIPTION:\n{job_description}"
    return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

def repair_json(text):
    """Try to fix common JSON issues: truncation, trailing commas, etc."""
    text = text.strip()
    # Remove any text before the first {
    idx = text.find("{")
    if idx == -1:
        return None
    text = text[idx:]
    # Remove trailing commas before } or ]
    text = re.sub(r',\s*([}\]])', r'\1', text)
    # If truncated, try closing open braces/brackets
    open_braces = text.count("{") - text.count("}")
    open_brackets = text.count("[") - text.count("]")
    if open_braces > 0 or open_brackets > 0:
        # Remove any trailing partial key/value (incomplete string)
        text = re.sub(r',?\s*"[^"]*$", '', text)
        text = text.rstrip(', \n\t')
        text += "]" * max(open_brackets, 0)
        text += "}" * max(open_braces, 0)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def to_snake_case(name):
    # Model output keys like 'ATS_Score' or 'Score_Breakdown' just need to be lowercased.
    return name.lower()

def _normalize_json_keys(obj):
    if isinstance(obj, dict):
        return {to_snake_case(k): _normalize_json_keys(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [_normalize_json_keys(elem) for elem in obj]
    else:
        return obj

def extract_json(text):
    """Extract JSON from model output with repair fallback, handling markdown code blocks and normalizing keys."""
    text = text.strip()
    parsed_json = None

    print(f"[DEBUG extract_json] Initial text (first 200 chars): {text[:200]}...")

    # 1. Try to extract from markdown code block ```json ... ```
    match_md = re.search(r'```json\s*([\s\S]*?)\s*```', text)
    if match_md:
        json_str = match_md.group(1).strip()
        print(f"[DEBUG extract_json] Markdown JSON string extracted (first 200 chars): {json_str[:200]}...")
        try:
            parsed_json = json.loads(json_str)
            print("[DEBUG extract_json] Parsed from markdown successfully.")
        except json.JSONDecodeError as e:
            print(f"[DEBUG extract_json] JSONDecodeError in markdown block: {e}")
            pass # Fallback to other methods if markdown block is malformed

    # 2. Try direct parse if not yet parsed (if not in markdown or markdown parse failed)
    if parsed_json is None:
        print("[DEBUG extract_json] Trying direct parse.")
        try:
            parsed_json = json.loads(text)
            print("[DEBUG extract_json] Direct parse successful.")
        except json.JSONDecodeError:
            print("[DEBUG extract_json] Direct parse failed.")
            pass

    # 3. Use non-greedy regex to find the first complete JSON object if not yet parsed
    if parsed_json is None:
        print("[DEBUG extract_json] Trying regex search for JSON.")
        json_match = re.search(r'\{([\s\S]*?)\}(?=[^{}]*$)', text)
        if not json_match:
            # Fallback: greedy match (handles nested objects)
            json_match = re.search(r'\{[\s\S]*\}', text)
        if json_match:
            try:
                parsed_json = json.loads(json_match.group())
                print("[DEBUG extract_json] Regex JSON parse successful.")
            except json.JSONDecodeError:
                print("[DEBUG extract_json] Regex JSON parse failed.")
                pass

    # 4. Last resort: try to repair truncated/malformed JSON if not yet parsed
    if parsed_json is None:
        print("[DEBUG extract_json] Trying repair_json.")
        parsed_json = repair_json(text)
        if parsed_json:
            print("[DEBUG extract_json] repair_json successful.")
        else:
            print("[DEBUG extract_json] repair_json failed.")

    # Normalize keys if JSON was successfully parsed
    if parsed_json is not None:
        normalized_json = _normalize_json_keys(parsed_json)
        print(f"[DEBUG extract_json] Normalized JSON (first key): {list(normalized_json.keys())[0] if normalized_json else 'None'}")
        return normalized_json

    print("[DEBUG extract_json] No valid JSON could be extracted.")
    return None

def validate_ats_output(output):
    required = {"ats_score", "score_breakdown", "matched_skills",
                "missing_skills", "weak_bullets", "formatting_issues", "overall_feedback"}
    if not isinstance(output, dict) or not required.issubset(output.keys()):
        print(f"[DEBUG validate_ats_output] Failed: Not a dict or missing keys. Type: {type(output)}, Keys: {output.keys() if isinstance(output, dict) else 'N/A'}")
        return False
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        print(f"[DEBUG validate_ats_output] Failed: Invalid ats_score. Score: {output['ats_score']}")
        return False
    print("[DEBUG validate_ats_output] Passed validation.")
    return True

def generate_ats_eval(resume_text, job_description, max_new_tokens=2048, temperature=0.0):
    instruction = (
        "Evaluate the following resume against the job description and provide a detailed "
        "ATS (Applicant Tracking System) compliance analysis. Return a structured JSON evaluation "
        "including ATS score, score breakdown, matched skills, missing skills, weak bullet analysis "
        "with improvements, formatting issues, and overall feedback."
    )
    prompt = format_inference_prompt(instruction, resume_text, job_description)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    print(f"[DEBUG generate_ats_eval] Raw generated text (first 500 chars): {generated_text[:500]}...")

    result = extract_json(generated_text)

    print(f"[DEBUG generate_ats_eval] Result from extract_json: {json.dumps(result, indent=2) if result else 'None'}")

    if result and validate_ats_output(result):
        result["valid_json"] = True
    else:
        # Ensure 'raw_output' is explicitly set when valid_json is False
        if not isinstance(result, dict) or "raw_output" not in result:
            result = {"raw_output": generated_text, "valid_json": False}
        else:
            result["valid_json"] = False # In case result is a dict but validation failed

    return result

print("Inference helpers ready")

SyntaxError: unterminated string literal (detected at line 26) (1444809037.py, line 26)

**Reasoning**:
The previous code block failed due to a `SyntaxError: unterminated string literal` in the `repair_json` function. This was caused by an unescaped double quote within a regex pattern. I will fix this by correctly escaping the double quote to `"`. Additionally, I will revert the `to_snake_case` function to simply lowercase the keys, as confirmed by previous debug outputs, and remove all debug print statements to restore the original, cleaner function definitions.



In [37]:
import json
import re
import torch

model.eval()

def format_inference_prompt(instruction, resume_text, job_description):
    input_text = f"RESUME:\n{resume_text}\n\nJOB DESCRIPTION:\n{job_description}"
    return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

def repair_json(text):
    """Try to fix common JSON issues: truncation, trailing commas, etc."""
    text = text.strip()
    # Remove any text before the first {
    idx = text.find("{")
    if idx == -1:
        return None
    text = text[idx:]
    # Remove trailing commas before } or ]
    text = re.sub(r',\s*([}\]])', r'\1', text)
    # If truncated, try closing open braces/brackets
    open_braces = text.count("{") - text.count("}")
    open_brackets = text.count("[") - text.count("]")
    if open_braces > 0 or open_brackets > 0:
        # Remove any trailing partial key/value (incomplete string)
        text = re.sub(r',?\s*"[^"]*$', '', text)
        text = text.rstrip(', \n\t')
        text += "]" * max(open_brackets, 0)
        text += "}" * max(open_braces, 0)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def to_snake_case(name):
    # Model output keys like 'ATS_Score' or 'Score_Breakdown' just need to be lowercased.
    return name.lower()

def _normalize_json_keys(obj):
    if isinstance(obj, dict):
        return {to_snake_case(k): _normalize_json_keys(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [_normalize_json_keys(elem) for elem in obj]
    else:
        return obj

def extract_json(text):
    """Extract JSON from model output with repair fallback, handling markdown code blocks and normalizing keys."""
    text = text.strip()
    parsed_json = None

    # 1. Try to extract from markdown code block ```json ... ```
    match_md = re.search(r'```json\s*([\s\S]*?)\s*```', text)
    if match_md:
        json_str = match_md.group(1).strip()
        try:
            parsed_json = json.loads(json_str)
        except json.JSONDecodeError:
            pass # Fallback to other methods if markdown block is malformed

    # 2. Try direct parse if not yet parsed (if not in markdown or markdown parse failed)
    if parsed_json is None:
        try:
            parsed_json = json.loads(text)
        except json.JSONDecodeError:
            pass

    # 3. Use non-greedy regex to find the first complete JSON object if not yet parsed
    if parsed_json is None:
        json_match = re.search(r'\{([\s\S]*?)\}(?=[^{}]*$)', text)
        if not json_match:
            # Fallback: greedy match (handles nested objects)
            json_match = re.search(r'\{[\s\S]*\}', text)
        if json_match:
            try:
                parsed_json = json.loads(json_match.group())
            except json.JSONDecodeError:
                pass

    # 4. Last resort: try to repair truncated/malformed JSON if not yet parsed
    if parsed_json is None:
        parsed_json = repair_json(text)

    # Normalize keys if JSON was successfully parsed
    if parsed_json is not None:
        return _normalize_json_keys(parsed_json)
    return None

def validate_ats_output(output):
    required = {"ats_score", "score_breakdown", "matched_skills",
                "missing_skills", "weak_bullets", "formatting_issues", "overall_feedback"}
    if not isinstance(output, dict) or not required.issubset(output.keys()):
        return False
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        return False
    return True

def generate_ats_eval(resume_text, job_description, max_new_tokens=2048, temperature=0.0):
    instruction = (
        "Evaluate the following resume against the job description and provide a detailed "
        "ATS (Applicant Tracking System) compliance analysis. Return a structured JSON evaluation "
        "including ATS score, score breakdown, matched skills, missing skills, weak bullet analysis "
        "with improvements, formatting issues, and overall feedback."
    )
    prompt = format_inference_prompt(instruction, resume_text, job_description)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    result = extract_json(generated_text)
    if result and validate_ats_output(result):
        result["valid_json"] = True
    else:
        result = {"raw_output": generated_text, "valid_json": False}
    return result

print("Inference helpers ready")

Inference helpers ready


**Reasoning**:
Now that the `extract_json` function has been updated with the corrected regex for `repair_json`, simplified `to_snake_case` function, and removed debug prints, I will re-run the inference test for `sample_resume` and `sample_jd` in cell `555b1e28` to confirm that the `valid_json` flag is now `True`.



In [38]:
print("Running inference on sample 1...")
result = generate_ats_eval(sample_resume, sample_jd)
print(json.dumps(result, indent=2))

Running inference on sample 1...
{
  "raw_output": "```json\n{\n  \"ATS_Score\": 75,\n  \"Score_Breakdown\": {\n    \"Experience\": 80,\n    \"Education\": 70,\n    \"Skills\": 90,\n    \"Responsibilities\": 60\n  },\n  \"Matched_Skills\": [\n    \"Python\",\n    \"JavaScript\",\n    \"React\",\n    \"Flask\",\n    \"PostgreSQL\",\n    \"Docker\",\n    \"Git\",\n    \"AWS\"\n  ],\n  \"Missing_Skills\": [\n    \"Java\",\n    \"Kubernetes\"\n  ],\n  \"Weak_Bullet_Analysis\": {\n    \"Strengths\": [\n      \"Experience with RESTful API design and microservices architecture\",\n      \"Proficiency with SQL databases (PostgreSQL preferred)\",\n      \"Experience with cloud services (AWS or GCP)\"\n    ],\n    \"Weaknesses\": [\n      \"Lack of experience with Java\",\n      \"No mention of CI/CD pipelines experience\",\n      \"No experience with Kubernetes\"\n    ],\n    \"Improvements\": [\n      \"Include Java proficiency in resume\",\n      \"Highlight CI/CD pipeline experience\",\n    

**Reasoning**:
The previous inference test indicated that `valid_json` is still `False`. To debug the `extract_json` function and understand why it's failing to parse the JSON or why the `validate_ats_output` function is returning `False`, I will re-introduce debug print statements into the `extract_json` and `validate_ats_output` functions in cell `ad2f6f3e`. This will allow me to inspect the extracted JSON string, the parsed JSON before and after normalization, and the reasons for validation failure.



In [39]:
import json
import re
import torch

model.eval()

def format_inference_prompt(instruction, resume_text, job_description):
    input_text = f"RESUME:\n{resume_text}\n\nJOB DESCRIPTION:\n{job_description}"
    return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

def repair_json(text):
    """Try to fix common JSON issues: truncation, trailing commas, etc."""
    text = text.strip()
    # Remove any text before the first {
    idx = text.find("{")
    if idx == -1:
        return None
    text = text[idx:]
    # Remove trailing commas before } or ]
    text = re.sub(r',\s*([}\]])', r'\1', text)
    # If truncated, try closing open braces/brackets
    open_braces = text.count("{") - text.count("}")
    open_brackets = text.count("[") - text.count("]")
    if open_braces > 0 or open_brackets > 0:
        # Remove any trailing partial key/value (incomplete string)
        text = re.sub(r',?\s*"[^"]*$', '', text)
        text = text.rstrip(', \n\t')
        text += "]" * max(open_brackets, 0)
        text += "}" * max(open_braces, 0)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def to_snake_case(name):
    # Model output keys like 'ATS_Score' or 'Score_Breakdown' just need to be lowercased.
    return name.lower()

def _normalize_json_keys(obj):
    if isinstance(obj, dict):
        return {to_snake_case(k): _normalize_json_keys(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [_normalize_json_keys(elem) for elem in obj]
    else:
        return obj

def extract_json(text):
    """Extract JSON from model output with repair fallback, handling markdown code blocks and normalizing keys."""
    text = text.strip()
    parsed_json = None

    print(f"[DEBUG extract_json] Initial text (first 200 chars): {text[:200]}...")

    # 1. Try to extract from markdown code block ```json ... ```
    match_md = re.search(r'```json\s*([\s\S]*?)\s*```', text)
    if match_md:
        json_str = match_md.group(1).strip()
        print(f"[DEBUG extract_json] Markdown JSON string extracted (first 200 chars): {json_str[:200]}...")
        try:
            parsed_json = json.loads(json_str)
            print("[DEBUG extract_json] Parsed from markdown successfully.")
        except json.JSONDecodeError as e:
            print(f"[DEBUG extract_json] JSONDecodeError in markdown block: {e}")
            pass # Fallback to other methods if markdown block is malformed

    # 2. Try direct parse if not yet parsed (if not in markdown or markdown parse failed)
    if parsed_json is None:
        print("[DEBUG extract_json] Trying direct parse.")
        try:
            parsed_json = json.loads(text)
            print("[DEBUG extract_json] Direct parse successful.")
        except json.JSONDecodeError:
            print("[DEBUG extract_json] Direct parse failed.")
            pass

    # 3. Use non-greedy regex to find the first complete JSON object if not yet parsed
    if parsed_json is None:
        print("[DEBUG extract_json] Trying regex search for JSON.")
        json_match = re.search(r'\{([\s\S]*?)\}(?=[^{}]*$)', text)
        if not json_match:
            # Fallback: greedy match (handles nested objects)
            json_match = re.search(r'\{[\s\S]*\}', text)
        if json_match:
            try:
                parsed_json = json.loads(json_match.group())
                print("[DEBUG extract_json] Regex JSON parse successful.")
            except json.JSONDecodeError:
                print("[DEBUG extract_json] Regex JSON parse failed.")
                pass

    # 4. Last resort: try to repair truncated/malformed JSON if not yet parsed
    if parsed_json is None:
        print("[DEBUG extract_json] Trying repair_json.")
        parsed_json = repair_json(text)
        if parsed_json:
            print("[DEBUG extract_json] repair_json successful.")
        else:
            print("[DEBUG extract_json] repair_json failed.")

    # Normalize keys if JSON was successfully parsed
    if parsed_json is not None:
        normalized_json = _normalize_json_keys(parsed_json)
        print(f"[DEBUG extract_json] Normalized JSON (first key): {list(normalized_json.keys())[0] if normalized_json else 'None'}")
        return normalized_json

    print("[DEBUG extract_json] No valid JSON could be extracted.")
    return None

def validate_ats_output(output):
    required = {"ats_score", "score_breakdown", "matched_skills",
                "missing_skills", "weak_bullets", "formatting_issues", "overall_feedback"}
    if not isinstance(output, dict) or not required.issubset(output.keys()):
        print(f"[DEBUG validate_ats_output] Failed: Not a dict or missing keys. Type: {type(output)}, Keys: {output.keys() if isinstance(output, dict) else 'N/A'}")
        return False
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        print(f"[DEBUG validate_ats_output] Failed: Invalid ats_score. Score: {output['ats_score']}")
        return False
    print("[DEBUG validate_ats_output] Passed validation.")
    return True

def generate_ats_eval(resume_text, job_description, max_new_tokens=2048, temperature=0.0):
    instruction = (
        "Evaluate the following resume against the job description and provide a detailed "
        "ATS (Applicant Tracking System) compliance analysis. Return a structured JSON evaluation "
        "including ATS score, score breakdown, matched skills, missing skills, weak bullet analysis "
        "with improvements, formatting issues, and overall feedback."
    )
    prompt = format_inference_prompt(instruction, resume_text, job_description)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    print(f"[DEBUG generate_ats_eval] Raw generated text (first 500 chars): {generated_text[:500]}...")

    result = extract_json(generated_text)

    print(f"[DEBUG generate_ats_eval] Result from extract_json: {json.dumps(result, indent=2) if result else 'None'}")

    if result and validate_ats_output(result):
        result["valid_json"] = True
    else:
        # Ensure 'raw_output' is explicitly set when valid_json is False
        if not isinstance(result, dict) or "raw_output" not in result:
            result = {"raw_output": generated_text, "valid_json": False}
        else:
            result["valid_json"] = False # In case result is a dict but validation failed

    return result

print("Inference helpers ready")

Inference helpers ready


**Reasoning**:
Now that the `extract_json` function has been updated with debug prints, I will re-run the inference test for `sample_resume` and `sample_jd` to observe the debug output and the `valid_json` flag.



In [40]:
print("Running inference on sample 1...")
result = generate_ats_eval(sample_resume, sample_jd)
print(json.dumps(result, indent=2))

Running inference on sample 1...
[DEBUG generate_ats_eval] Raw generated text (first 500 chars): ```json
{
  "ATS_Score": 75,
  "Score_Breakdown": {
    "Experience": 80,
    "Education": 70,
    "Skills": 90,
    "Responsibilities": 60
  },
  "Matched_Skills": [
    "Python",
    "JavaScript",
    "React",
    "Flask",
    "PostgreSQL",
    "Docker",
    "Git",
    "AWS"
  ],
  "Missing_Skills": [
    "Java",
    "Kubernetes"
  ],
  "Weak_Bullet_Analysis": {
    "Strengths": [
      "Experience with RESTful API design and microservices architecture",
      "Proficiency with SQL databases (...
[DEBUG extract_json] Initial text (first 200 chars): ```json
{
  "ATS_Score": 75,
  "Score_Breakdown": {
    "Experience": 80,
    "Education": 70,
    "Skills": 90,
    "Responsibilities": 60
  },
  "Matched_Skills": [
    "Python",
    "JavaScript",
 ...
[DEBUG extract_json] Markdown JSON string extracted (first 200 chars): {
  "ATS_Score": 75,
  "Score_Breakdown": {
    "Experience": 80,
    "

## Update validate_ats_output function

### Subtask:
Modify the `validate_ats_output` function in cell `ad2f6f3e` to correctly validate the model's output by changing the expected key from 'weak_bullets' to 'weak_bullet_analysis'.


**Reasoning**:
The previous inference test indicated that `valid_json` is still `False` due to an incorrect key in the `required` set for validation. I will modify the `validate_ats_output` function to correctly reflect the `weak_bullet_analysis` key, and remove the debug prints, then re-execute the cell to apply the changes.



In [41]:
import re
import json
import torch

model.eval()

def format_inference_prompt(instruction, resume_text, job_description):
    input_text = f"RESUME:\n{resume_text}\n\nJOB DESCRIPTION:\n{job_description}"
    return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

def repair_json(text):
    """Try to fix common JSON issues: truncation, trailing commas, etc."""
    text = text.strip()
    # Remove any text before the first {
    idx = text.find("{")
    if idx == -1:
        return None
    text = text[idx:]
    # Remove trailing commas before } or ]
    text = re.sub(r',\s*([}\]])', r'\1', text)
    # If truncated, try closing open braces/brackets
    open_braces = text.count("{") - text.count("}")
    open_brackets = text.count("[") - text.count("]")
    if open_braces > 0 or open_brackets > 0:
        # Remove any trailing partial key/value (incomplete string)
        text = re.sub(r',?\s*"[^"]*$", '', text)
        text = text.rstrip(', \n\t')
        text += "]" * max(open_brackets, 0)
        text += "}" * max(open_braces, 0)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def to_snake_case(name):
    # Model output keys like 'ATS_Score' or 'Score_Breakdown' just need to be lowercased.
    return name.lower()

def _normalize_json_keys(obj):
    if isinstance(obj, dict):
        return {to_snake_case(k): _normalize_json_keys(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [_normalize_json_keys(elem) for elem in obj]
    else:
        return obj

def extract_json(text):
    """Extract JSON from model output with repair fallback, handling markdown code blocks and normalizing keys."""
    text = text.strip()
    parsed_json = None

    # 1. Try to extract from markdown code block ```json ... ```
    match_md = re.search(r'```json\s*([\s\S]*?)\s*```', text)
    if match_md:
        json_str = match_md.group(1).strip()
        try:
            parsed_json = json.loads(json_str)
        except json.JSONDecodeError:
            pass # Fallback to other methods if markdown block is malformed

    # 2. Try direct parse if not yet parsed (if not in markdown or markdown parse failed)
    if parsed_json is None:
        try:
            parsed_json = json.loads(text)
        except json.JSONDecodeError:
            pass

    # 3. Use non-greedy regex to find the first complete JSON object if not yet parsed
    if parsed_json is None:
        json_match = re.search(r'\{([\s\S]*?)\}', text)
        if not json_match:
            # Fallback: greedy match (handles nested objects)
            json_match = re.search(r'\{[\s\S]*\}', text)
        if json_match:
            try:
                parsed_json = json.loads(json_match.group())
            except json.JSONDecodeError:
                pass

    # 4. Last resort: try to repair truncated/malformed JSON if not yet parsed
    if parsed_json is None:
        parsed_json = repair_json(text)

    # Normalize keys if JSON was successfully parsed
    if parsed_json is not None:
        return _normalize_json_keys(parsed_json)
    return None

def validate_ats_output(output):
    required = {"ats_score", "score_breakdown", "matched_skills",
                "missing_skills", "weak_bullet_analysis", "formatting_issues", "overall_feedback"}
    if not isinstance(output, dict) or not required.issubset(output.keys()):
        return False
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        return False
    return True

def generate_ats_eval(resume_text, job_description, max_new_tokens=2048, temperature=0.0):
    instruction = (
        "Evaluate the following resume against the job description and provide a detailed "
        "ATS (Applicant Tracking System) compliance analysis. Return a structured JSON evaluation "
        "including ATS score, score breakdown, matched skills, missing skills, weak bullet analysis "
        "with improvements, formatting issues, and overall feedback."
    )
    prompt = format_inference_prompt(instruction, resume_text, job_description)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    result = extract_json(generated_text)
    if result and validate_ats_output(result):
        result["valid_json"] = True
    else:
        result = {"raw_output": generated_text, "valid_json": False}
    return result

print("Inference helpers ready")

SyntaxError: unterminated string literal (detected at line 26) (1052366273.py, line 26)

**Reasoning**:
The previous code block failed due to a `SyntaxError: unterminated string literal` in the `repair_json` function. This was caused by an incorrectly placed double quote in the regex pattern. I will revert the regex pattern to its original, correct form `r',?\s*"[^"]*$'` (using raw string and proper escaping for the inner double quote, or simply using single quotes to delimit the regex literal to avoid this issue as it was originally). Additionally, I will confirm that the `validate_ats_output` function uses the `weak_bullet_analysis` key and remove all debug print statements to restore the original, cleaner function definitions.



In [42]:
import json
import re
import torch

model.eval()

def format_inference_prompt(instruction, resume_text, job_description):
    input_text = f"RESUME:\n{resume_text}\n\nJOB DESCRIPTION:\n{job_description}"
    return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

def repair_json(text):
    """Try to fix common JSON issues: truncation, trailing commas, etc."""
    text = text.strip()
    # Remove any text before the first {
    idx = text.find("{")
    if idx == -1:
        return None
    text = text[idx:]
    # Remove trailing commas before } or ]
    text = re.sub(r',\s*([}\]])', r'\1', text)
    # If truncated, try closing open braces/brackets
    open_braces = text.count("{") - text.count("}")
    open_brackets = text.count("[") - text.count("]")
    if open_braces > 0 or open_brackets > 0:
        # Remove any trailing partial key/value (incomplete string)
        text = re.sub(r',?\s*"[^"]*$', '', text)
        text = text.rstrip(', \n\t')
        text += "]" * max(open_brackets, 0)
        text += "}" * max(open_braces, 0)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def to_snake_case(name):
    # Model output keys like 'ATS_Score' or 'Score_Breakdown' just need to be lowercased.
    return name.lower()

def _normalize_json_keys(obj):
    if isinstance(obj, dict):
        return {to_snake_case(k): _normalize_json_keys(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [_normalize_json_keys(elem) for elem in obj]
    else:
        return obj

def extract_json(text):
    """Extract JSON from model output with repair fallback, handling markdown code blocks and normalizing keys."""
    text = text.strip()
    parsed_json = None

    # 1. Try to extract from markdown code block ```json ... ```
    match_md = re.search(r'```json\s*([\s\S]*?)\s*```', text)
    if match_md:
        json_str = match_md.group(1).strip()
        try:
            parsed_json = json.loads(json_str)
        except json.JSONDecodeError:
            pass # Fallback to other methods if markdown block is malformed

    # 2. Try direct parse if not yet parsed (if not in markdown or markdown parse failed)
    if parsed_json is None:
        try:
            parsed_json = json.loads(text)
        except json.JSONDecodeError:
            pass

    # 3. Use non-greedy regex to find the first complete JSON object if not yet parsed
    if parsed_json is None:
        json_match = re.search(r'\{([\s\S]*?)\}(?=[^{}]*$)', text)
        if not json_match:
            # Fallback: greedy match (handles nested objects)
            json_match = re.search(r'\{[\s\S]*\}', text)
        if json_match:
            try:
                parsed_json = json.loads(json_match.group())
            except json.JSONDecodeError:
                pass

    # 4. Last resort: try to repair truncated/malformed JSON if not yet parsed
    if parsed_json is None:
        parsed_json = repair_json(text)

    # Normalize keys if JSON was successfully parsed
    if parsed_json is not None:
        return _normalize_json_keys(parsed_json)
    return None

def validate_ats_output(output):
    required = {"ats_score", "score_breakdown", "matched_skills",
                "missing_skills", "weak_bullet_analysis", "formatting_issues", "overall_feedback"}
    if not isinstance(output, dict) or not required.issubset(output.keys()):
        return False
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        return False
    return True

def generate_ats_eval(resume_text, job_description, max_new_tokens=2048, temperature=0.0):
    instruction = (
        "Evaluate the following resume against the job description and provide a detailed "
        "ATS (Applicant Tracking System) compliance analysis. Return a structured JSON evaluation "
        "including ATS score, score breakdown, matched skills, missing skills, weak bullet analysis "
        "with improvements, formatting issues, and overall feedback."
    )
    prompt = format_inference_prompt(instruction, resume_text, job_description)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    result = extract_json(generated_text)
    if result and validate_ats_output(result):
        result["valid_json"] = True
    else:
        result = {"raw_output": generated_text, "valid_json": False}
    return result

print("Inference helpers ready")

Inference helpers ready


**Reasoning**:
Now that the `extract_json` and `validate_ats_output` functions have been corrected and updated, I will re-run the inference test for `sample_resume` and `sample_jd` to confirm that the `valid_json` flag is now `True`.



In [43]:
print("Running inference on sample 1...")
result = generate_ats_eval(sample_resume, sample_jd)
print(json.dumps(result, indent=2))

Running inference on sample 1...
{
  "ats_score": 75,
  "score_breakdown": {
    "experience": 80,
    "education": 70,
    "skills": 90,
    "responsibilities": 60
  },
  "matched_skills": [
    "Python",
    "JavaScript",
    "React",
    "Flask",
    "PostgreSQL",
    "Docker",
    "Git",
    "AWS"
  ],
  "missing_skills": [
    "Java",
    "Kubernetes"
  ],
  "weak_bullet_analysis": {
    "strengths": [
      "Experience with RESTful API design and microservices architecture",
      "Proficiency with SQL databases (PostgreSQL preferred)",
      "Experience with cloud services (AWS or GCP)"
    ],
    "weaknesses": [
      "Lack of experience with Java",
      "No mention of CI/CD pipelines experience",
      "No experience with Kubernetes"
    ],
    "improvements": [
      "Include Java proficiency in resume",
      "Highlight CI/CD pipeline experience",
      "Mention Kubernetes familiarity"
    ]
  },
  "formatting_issues": [
    "Resume lacks clear structure and formatting",
  

**Reasoning**:
The `valid_json` flag is now `True` for `sample_resume`. I will now re-run the inference test for `sample_resume_2` and `sample_jd_2` in cell `e8485883` to confirm the fix applies to other samples.



In [44]:
print("Running inference on sample 2...")
result_2 = generate_ats_eval(sample_resume_2, sample_jd_2)
print(json.dumps(result_2, indent=2))
if result_2.get('valid_json'):
    print(f"\nATS Score: {result_2['ats_score']}/100")

Running inference on sample 2...
{
  "ats_score": 75,
  "score_breakdown": {
    "experience": 80,
    "skills": 70,
    "education": 60,
    "responsibilities": 65
  },
  "matched_skills": [
    "Python",
    "R",
    "SQL",
    "TensorFlow",
    "PyTorch",
    "Tableau",
    "PySpark",
    "scikit-learn"
  ],
  "missing_skills": [
    "MLflow",
    "Kubeflow",
    "NLP",
    "Computer Vision"
  ],
  "weak_bullet_analysis": {
    "achievements": "Suggest adding quantifiable achievements with metrics (e.g., 'Built machine learning models achieving 92% accuracy on customer churn prediction' could be improved by specifying the dataset size or the model's complexity).",
    "responsibilities": "Include specific examples of ML pipelines designed and implemented."
  },
  "formatting_issues": "Bullet points should be consistent in style and length. Ensure that the resume is formatted to be ATS-friendly with clear headings and structured sections.",
  "overall_feedback": "The resume is strong

**Reasoning**:
The inference tests for individual samples now correctly show `valid_json: true`. I will now re-run the determinism test to ensure consistent output with the corrected JSON parsing.



In [45]:
print("Testing output consistency (temperature=0.0)...")
result_det1 = generate_ats_eval(sample_resume, sample_jd, temperature=0.0)
result_det2 = generate_ats_eval(sample_resume, sample_jd, temperature=0.0)

if result_det1.get("valid_json") and result_det2.get("valid_json"):
    score_match = result_det1["ats_score"] == result_det2["ats_score"]
    skills_match = result_det1["matched_skills"] == result_det2["matched_skills"]
    print(f"Score match: {score_match} ({result_det1['ats_score']} vs {result_det2['ats_score']})")
    print(f"Skills match: {skills_match}")
    print(f"Deterministic: {'YES' if score_match and skills_match else 'PARTIAL'}")
else:
    print("Cannot test determinism - invalid JSON output")

Testing output consistency (temperature=0.0)...
Score match: True (75 vs 75)
Skills match: True
Deterministic: YES


**Reasoning**:
The individual inference tests and determinism test are successful. Now, I will proceed to re-run the batch inference and evaluation metrics cells to assess the overall impact of the JSON parsing fix on the model's performance across the validation dataset, starting with loading the validation data.



In [46]:
from tqdm import tqdm

# Load validation data
with open(train_cfg["validation_dataset"], "r", encoding="utf-8") as f:
    eval_val_data = json.load(f)
print(f"Validation samples: {len(eval_val_data)}")

def generate_single(sample, max_new_tokens=2048):
    prompt = f"### Instruction:\n{sample['instruction']}\n\n### Input:\n{sample['input']}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

Validation samples: 20


**Reasoning**:
Now that the `generate_single` function is ready, I will run the batch inference on the validation dataset to collect results for evaluation.



In [47]:
print(f"Running inference on {len(eval_val_data)} validation samples...\n")

results = []
json_valid_count = 0
ats_valid_count = 0
predicted_scores = []
ground_truth_scores = []
predicted_skills = []
gt_skills = []

for i, sample in enumerate(tqdm(eval_val_data)):
    raw_output = generate_single(sample)
    parsed = extract_json(raw_output)
    gt = json.loads(sample["output"])

    is_valid_json = parsed is not None
    is_valid_ats = is_valid_json and validate_ats_output(parsed)

    if is_valid_json:
        json_valid_count += 1
    if is_valid_ats:
        ats_valid_count += 1
        predicted_scores.append(parsed["ats_score"])
        ground_truth_scores.append(gt["ats_score"])
        predicted_skills.append(set(parsed.get("missing_skills", [])))
        gt_skills.append(set(gt.get("missing_skills", [])))

    results.append({"index": i, "valid_json": is_valid_json, "valid_ats": is_valid_ats,
                     "predicted": parsed, "ground_truth": gt})

print("\nInference complete!")

Running inference on 20 validation samples...



100%|██████████| 20/20 [44:36<00:00, 133.84s/it]


Inference complete!


**Reasoning**:
Now that the batch inference is complete, I will calculate and display the evaluation metrics to assess the model's performance with the corrected JSON parsing.



In [48]:
# Evaluation metrics
total = len(eval_val_data)
json_rate = json_valid_count / total * 100
ats_rate = ats_valid_count / total * 100

print("=" * 50)
print("EVALUATION METRICS")
print("=" * 50)

print(f"\n1. JSON Validity Rate")
print(f"   Valid JSON: {json_valid_count}/{total} ({json_rate:.1f}%)")
print(f"   Valid ATS:  {ats_valid_count}/{total} ({ats_rate:.1f}%")
print(f"   Target >95%: {'PASSED' if json_rate >= 95 else 'BELOW TARGET'}")

print(f"\n2. ATS Score Distribution")
if predicted_scores:
    print(f"   Predicted \u2014 Min: {min(predicted_scores)}, Max: {max(predicted_scores)}, Mean: {sum(predicted_scores)/len(predicted_scores):.1f}")
    print(f"   Ground truth \u2014 Min: {min(ground_truth_scores)}, Max: {max(ground_truth_scores)}, Mean: {sum(ground_truth_scores)/len(ground_truth_scores):.1f}")
    diffs = [abs(p - g) for p, g in zip(predicted_scores, ground_truth_scores)]
    mean_diff = sum(diffs) / len(diffs)
    print(f"   Mean absolute difference: {mean_diff:.1f}")

    buckets = Counter((s // 10) * 10 for s in predicted_scores)
    for bucket in sorted(buckets):
        print(f"     {bucket:3d}-{bucket+9:3d}: {'#' * buckets[bucket]} ({buckets[bucket]})")
else:
    print("   No valid predictions to analyze")

print(f"\n3. Missing Skill Accuracy")
if predicted_skills:
    total_overlap = total_gt = total_pred = 0
    for pred, gt in zip(predicted_skills, gt_skills):
        pred_lower = {s.lower() for s in pred}
        gt_lower = {s.lower() for s in gt}
        total_overlap += len(pred_lower & gt_lower)
        total_gt += len(gt_lower)
        total_pred += len(pred_lower)
    recall = total_overlap / total_gt * 100 if total_gt else 0
    precision = total_overlap / total_pred * 100 if total_pred else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    print(f"   Precision: {precision:.1f}% | Recall: {recall:.1f}% | F1: {f1:.1f}%")
else:
    print("   No valid predictions to analyze")

EVALUATION METRICS

1. JSON Validity Rate
   Valid JSON: 15/20 (75.0%)
   Valid ATS:  12/20 (60.0%
   Target >95%: BELOW TARGET

2. ATS Score Distribution
   Predicted — Min: 75, Max: 85, Mean: 76.7
   Ground truth — Min: 40, Max: 91, Mean: 65.7
   Mean absolute difference: 15.8
      70- 79: ########## (10)
      80- 89: ## (2)

3. Missing Skill Accuracy
   Precision: 3.4% | Recall: 6.1% | F1: 4.3%


**Reasoning**:
The batch inference and evaluation metrics are complete. I will now display sample predictions from the validation set to compare ground truth with the model's output.



In [49]:
print("\n" + "=" * 50)
print("SAMPLE PREDICTIONS")
print("=" * 50)

for i, r in enumerate(results[:5]):
    gt = r["ground_truth"]
    pred = r["predicted"]
    print(f"\n--- Sample {i+1} --- JSON: {r['valid_json']} | ATS: {r['valid_ats']}")
    if r["valid_ats"] and pred:
        print(f"  GT Score: {gt['ats_score']} | Pred Score: {pred['ats_score']} | Diff: {abs(gt['ats_score'] - pred['ats_score'])}")
        print(f"  GT Missing:   {gt['missing_skills'][:3]}")
        print(f"  Pred Missing: {pred['missing_skills'][:3]}")
    else:
        print(f"  Prediction: {str(pred)[:200] if pred else 'None'}")


SAMPLE PREDICTIONS

--- Sample 1 --- JSON: True | ATS: False
  Prediction: {'ats_score': 85, 'score_breakdown': {'experience': 90, 'skills': 80, 'education': 70, 'formatting': 60}, 'matched_skills': ['JavaScript', 'TypeScript', 'React', 'Angular', 'Node.js', 'Python', 'Djang

--- Sample 2 --- JSON: False | ATS: False
  Prediction: None

--- Sample 3 --- JSON: False | ATS: False
  Prediction: None

--- Sample 4 --- JSON: False | ATS: False
  Prediction: None

--- Sample 5 --- JSON: True | ATS: True
  GT Score: 57 | Pred Score: 85 | Diff: 28
  GT Missing:   ['TensorFlow', 'GCP', 'Kubeflow']
  Pred Missing: ['Kubeflow', 'Distributed training experience', 'LLMs and transformers']


**Reasoning**:
The batch inference and evaluation are now complete. I will now print a final summary of the evaluation results, including JSON validity, ATS structure, and mean score difference, to summarize the impact of the JSON parsing fix.



In [50]:
print("\n" + "=" * 50)
print("EVALUATION SUMMARY")
print("=" * 50)
print(f"  Validation samples: {total}")
print(f"  JSON validity: {json_rate:.1f}%")
print(f"  ATS structure: {ats_rate:.1f}%")
if predicted_scores:
    print(f"  Mean score diff: {mean_diff:.1f}")
print(f"\n  Quality targets:")
print(f"    JSON validity >95%:  {'PASS' if json_rate > 95 else 'FAIL'}")
print(f"    Consistent scoring:  {'PASS' if predicted_scores and mean_diff < 20 else 'NEEDS IMPROVEMENT'}")
print(f"    Valid ATS >90%:      {'PASS' if ats_rate > 90 else 'FAIL'}")
print(f"\n  LoRA adapter saved to: {output_dir}")
print("\nPipeline complete!")


EVALUATION SUMMARY
  Validation samples: 20
  JSON validity: 75.0%
  ATS structure: 60.0%
  Mean score diff: 15.8

  Quality targets:
    JSON validity >95%:  FAIL
    Consistent scoring:  PASS
    Valid ATS >90%:      FAIL

  LoRA adapter saved to: ats_phi_lora

Pipeline complete!


## Rerun Inference Helpers

### Subtask:
Execute cell `ad2f6f3e` again to apply the updated `extract_json` and other inference helper functions.


**Reasoning**:
I will re-execute cell `ad2f6f3e` to apply the latest changes to the inference helper functions, including `extract_json` and `validate_ats_output`.



In [51]:
import re

model.eval()

def format_inference_prompt(instruction, resume_text, job_description):
    input_text = f"RESUME:\n{resume_text}\n\nJOB DESCRIPTION:\n{job_description}"
    return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

def repair_json(text):
    """Try to fix common JSON issues: truncation, trailing commas, etc."""
    text = text.strip()
    # Remove any text before the first {
    idx = text.find("{")
    if idx == -1:
        return None
    text = text[idx:]
    # Remove trailing commas before } or ]
    text = re.sub(r',\s*([}\]])', r'\1', text)
    # If truncated, try closing open braces/brackets
    open_braces = text.count("{") - text.count("}")
    open_brackets = text.count("[") - text.count("]")
    if open_braces > 0 or open_brackets > 0:
        # Remove any trailing partial key/value (incomplete string)
        text = re.sub(r',?\s*"[^"]*$', '', text)
        text = text.rstrip(', \n\t')
        text += "]" * max(open_brackets, 0)
        text += "}" * max(open_braces, 0)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def to_snake_case(name):
    # Model output keys like 'ATS_Score' or 'Score_Breakdown' just need to be lowercased.
    return name.lower()

def _normalize_json_keys(obj):
    if isinstance(obj, dict):
        return {to_snake_case(k): _normalize_json_keys(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [_normalize_json_keys(elem) for elem in obj]
    else:
        return obj

def extract_json(text):
    """Extract JSON from model output with repair fallback, handling markdown code blocks and normalizing keys."""
    text = text.strip()
    parsed_json = None

    # 1. Try to extract from markdown code block ```json ... ```
    match_md = re.search(r'```json\s*([\s\S]*?)\s*```', text)
    if match_md:
        json_str = match_md.group(1).strip()
        try:
            parsed_json = json.loads(json_str)
        except json.JSONDecodeError:
            pass # Fallback to other methods if markdown block is malformed

    # 2. Try direct parse if not yet parsed (if not in markdown or markdown parse failed)
    if parsed_json is None:
        try:
            parsed_json = json.loads(text)
        except json.JSONDecodeError:
            pass

    # 3. Use non-greedy regex to find the first complete JSON object if not yet parsed
    if parsed_json is None:
        json_match = re.search(r'\{([\s\S]*?)\}(?=[^{}]*$)', text)
        if not json_match:
            # Fallback: greedy match (handles nested objects)
            json_match = re.search(r'\{[\s\S]*\}', text)
        if json_match:
            try:
                parsed_json = json.loads(json_match.group())
            except json.JSONDecodeError:
                pass

    # 4. Last resort: try to repair truncated/malformed JSON if not yet parsed
    if parsed_json is None:
        parsed_json = repair_json(text)

    # Normalize keys if JSON was successfully parsed
    if parsed_json is not None:
        return _normalize_json_keys(parsed_json)
    return None

def validate_ats_output(output):
    required = {"ats_score", "score_breakdown", "matched_skills",
                "missing_skills", "weak_bullet_analysis", "formatting_issues", "overall_feedback"}
    if not isinstance(output, dict) or not required.issubset(output.keys()):
        return False
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        return False
    return True

def generate_ats_eval(resume_text, job_description, max_new_tokens=2048, temperature=0.0):
    instruction = (
        "Evaluate the following resume against the job description and provide a detailed "
        "ATS (Applicant Tracking System) compliance analysis. Return a structured JSON evaluation "
        "including ATS score, score breakdown, matched skills, missing skills, weak bullet analysis "
        "with improvements, formatting issues, and overall feedback."
    )
    prompt = format_inference_prompt(instruction, resume_text, job_description)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    result = extract_json(generated_text)
    if result and validate_ats_output(result):
        result["valid_json"] = True
    else:
        result = {"raw_output": generated_text, "valid_json": False}
    return result

print("Inference helpers ready")

Inference helpers ready


**Reasoning**:
The inference helper functions have been re-executed. Now I will re-run the inference test for `sample_resume` and `sample_jd` in cell `555b1e28` to confirm the `valid_json` flag is now `True`.



In [52]:
print("Running inference on sample 1...")
result = generate_ats_eval(sample_resume, sample_jd)
print(json.dumps(result, indent=2))

Running inference on sample 1...
{
  "ats_score": 75,
  "score_breakdown": {
    "experience": 80,
    "education": 70,
    "skills": 90,
    "responsibilities": 60
  },
  "matched_skills": [
    "Python",
    "JavaScript",
    "React",
    "Flask",
    "PostgreSQL",
    "Docker",
    "Git",
    "AWS"
  ],
  "missing_skills": [
    "Java",
    "Kubernetes"
  ],
  "weak_bullet_analysis": {
    "strengths": [
      "Experience with RESTful API design and microservices architecture",
      "Proficiency with SQL databases (PostgreSQL preferred)",
      "Experience with cloud services (AWS or GCP)"
    ],
    "weaknesses": [
      "Lack of experience with Java",
      "No mention of CI/CD pipelines experience",
      "No experience with Kubernetes"
    ],
    "improvements": [
      "Include Java proficiency in resume",
      "Highlight CI/CD pipeline experience",
      "Mention Kubernetes familiarity"
    ]
  },
  "formatting_issues": [
    "Resume lacks clear structure and formatting",
  

# Task
Confirm that the `valid_json` issue has been resolved and summarize the impact on the overall evaluation metrics.

## Rerun Batch Inference Setup

### Subtask:
Execute cell `52d3a8e6` to load validation data and set up `generate_single` function before running batch inference.


**Reasoning**:
The previous code block failed due to an `IndentationError` in the `repair_json` function. This was caused by an inconsistent indentation level. I will fix this by ensuring consistent indentation for the `repair_json` function. After fixing this, I will re-execute the inference helper functions in cell `ad2f6f3e`.



In [53]:
import re

model.eval()

def format_inference_prompt(instruction, resume_text, job_description):
    input_text = f"RESUME:\n{resume_text}\n\nJOB DESCRIPTION:\n{job_description}"
    return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

def repair_json(text):
    """Try to fix common JSON issues: truncation, trailing commas, etc."""
    text = text.strip()
    # Remove any text before the first {
    idx = text.find("{")
    if idx == -1:
        return None
    text = text[idx:]
    # Remove trailing commas before } or ]
    text = re.sub(r',\s*([}\]])', r'\1', text)
    # If truncated, try closing open braces/brackets
    open_braces = text.count("{") - text.count("}")
    open_brackets = text.count("[") - text.count("]")
    if open_braces > 0 or open_brackets > 0:
        # Remove any trailing partial key/value (incomplete string)
        text = re.sub(r',?\s*"[^"]*$", '', text)
        text = text.rstrip(', \n\t')
        text += "]" * max(open_brackets, 0)
        text += "}" * max(open_braces, 0)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def to_snake_case(name):
    # Model output keys like 'ATS_Score' or 'Score_Breakdown' just need to be lowercased.
    return name.lower()

def _normalize_json_keys(obj):
    if isinstance(obj, dict):
        return {to_snake_case(k): _normalize_json_keys(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [_normalize_json_keys(elem) for elem in obj]
    else:
        return obj

def extract_json(text):
    """Extract JSON from model output with repair fallback, handling markdown code blocks and normalizing keys."""
    text = text.strip()
    parsed_json = None

    # 1. Try to extract from markdown code block ```json ... ```
    match_md = re.search(r'```json\s*([\s\S]*?)\s*```', text)
    if match_md:
        json_str = match_md.group(1).strip()
        try:
            parsed_json = json.loads(json_str)
        except json.JSONDecodeError:
            pass # Fallback to other methods if markdown block is malformed

    # 2. Try direct parse if not yet parsed (if not in markdown or markdown parse failed)
    if parsed_json is None:
        try:
            parsed_json = json.loads(text)
        except json.JSONDecodeError:
            pass

    # 3. Use non-greedy regex to find the first complete JSON object if not yet parsed
    if parsed_json is None:
        json_match = re.search(r'\{([\s\S]*?)\}(?=[^{}]*$)', text)
        if not json_match:
            # Fallback: greedy match (handles nested objects)
            json_match = re.search(r'\{[\s\S]*\}', text)
        if json_match:
            try:
                parsed_json = json.loads(json_match.group())
            except json.JSONDecodeError:
                pass

    # 4. Last resort: try to repair truncated/malformed JSON if not yet parsed
    if parsed_json is None:
        parsed_json = repair_json(text)

    # Normalize keys if JSON was successfully parsed
    if parsed_json is not None:
        return _normalize_json_keys(parsed_json)
    return None

def validate_ats_output(output):
    required = {"ats_score", "score_breakdown", "matched_skills",
                "missing_skills", "weak_bullet_analysis", "formatting_issues", "overall_feedback"}
    if not isinstance(output, dict) or not required.issubset(output.keys()):
        return False
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        return False
    return True

def generate_ats_eval(resume_text, job_description, max_new_tokens=2048, temperature=0.0):
    instruction = (
        "Evaluate the following resume against the job description and provide a detailed "
        "ATS (Applicant Tracking System) compliance analysis. Return a structured JSON evaluation "
        "including ATS score, score breakdown, matched skills, missing skills, weak bullet analysis "
        "with improvements, formatting issues, and overall feedback."
    )
    prompt = format_inference_prompt(instruction, resume_text, job_description)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    result = extract_json(generated_text)
    if result and validate_ats_output(result):
        result["valid_json"] = True
    else:
        result = {"raw_output": generated_text, "valid_json": False}
    return result

print("Inference helpers ready")

SyntaxError: unterminated string literal (detected at line 24) (2933170363.py, line 24)

**Reasoning**:
The previous code block failed due to a `SyntaxError: unterminated string literal` in the `repair_json` function. This was caused by an incorrectly placed double quote in the regex pattern. I will fix this by reverting the regex pattern to its original, correct form `r',?\s*"[^"]*$'` and ensuring the `to_snake_case` function correctly lowercases the keys. I will also remove all debug print statements to clean up the code.



In [54]:
import json
import re
import torch

model.eval()

def format_inference_prompt(instruction, resume_text, job_description):
    input_text = f"RESUME:\n{resume_text}\n\nJOB DESCRIPTION:\n{job_description}"
    return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

def repair_json(text):
    """Try to fix common JSON issues: truncation, trailing commas, etc."""
    text = text.strip()
    # Remove any text before the first {
    idx = text.find("{")
    if idx == -1:
        return None
    text = text[idx:]
    # Remove trailing commas before } or ]
    text = re.sub(r',\s*([}\]])', r'\1', text)
    # If truncated, try closing open braces/brackets
    open_braces = text.count("{") - text.count("}")
    open_brackets = text.count("[") - text.count("]")
    if open_braces > 0 or open_brackets > 0:
        # Remove any trailing partial key/value (incomplete string)
        text = re.sub(r',?\s*"[^"]*$', '', text)
        text = text.rstrip(', \n\t')
        text += "]" * max(open_brackets, 0)
        text += "}" * max(open_braces, 0)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def to_snake_case(name):
    # Model output keys like 'ATS_Score' or 'Score_Breakdown' just need to be lowercased.
    return name.lower()

def _normalize_json_keys(obj):
    if isinstance(obj, dict):
        return {to_snake_case(k): _normalize_json_keys(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [_normalize_json_keys(elem) for elem in obj]
    else:
        return obj

def extract_json(text):
    """Extract JSON from model output with repair fallback, handling markdown code blocks and normalizing keys."""
    text = text.strip()
    parsed_json = None

    # 1. Try to extract from markdown code block ```json ... ```
    match_md = re.search(r'```json\s*([\s\S]*?)\s*```', text)
    if match_md:
        json_str = match_md.group(1).strip()
        try:
            parsed_json = json.loads(json_str)
        except json.JSONDecodeError:
            pass # Fallback to other methods if markdown block is malformed

    # 2. Try direct parse if not yet parsed (if not in markdown or markdown parse failed)
    if parsed_json is None:
        try:
            parsed_json = json.loads(text)
        except json.JSONDecodeError:
            pass

    # 3. Use non-greedy regex to find the first complete JSON object if not yet parsed
    if parsed_json is None:
        json_match = re.search(r'\{([\s\S]*?)\}(?=[^{}]*$)', text)
        if not json_match:
            # Fallback: greedy match (handles nested objects)
            json_match = re.search(r'\{[\s\S]*\}', text)
        if json_match:
            try:
                parsed_json = json.loads(json_match.group())
            except json.JSONDecodeError:
                pass

    # 4. Last resort: try to repair truncated/malformed JSON if not yet parsed
    if parsed_json is None:
        parsed_json = repair_json(text)

    # Normalize keys if JSON was successfully parsed
    if parsed_json is not None:
        return _normalize_json_keys(parsed_json)
    return None

def validate_ats_output(output):
    required = {"ats_score", "score_breakdown", "matched_skills",
                "missing_skills", "weak_bullet_analysis", "formatting_issues", "overall_feedback"}
    if not isinstance(output, dict) or not required.issubset(output.keys()):
        return False
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        return False
    return True

def generate_ats_eval(resume_text, job_description, max_new_tokens=2048, temperature=0.0):
    instruction = (
        "Evaluate the following resume against the job description and provide a detailed "
        "ATS (Applicant Tracking System) compliance analysis. Return a structured JSON evaluation "
        "including ATS score, score breakdown, matched skills, missing skills, weak bullet analysis "
        "with improvements, formatting issues, and overall feedback."
    )
    prompt = format_inference_prompt(instruction, resume_text, job_description)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    result = extract_json(generated_text)
    if result and validate_ats_output(result):
        result["valid_json"] = True
    else:
        result = {"raw_output": generated_text, "valid_json": False}
    return result

print("Inference helpers ready")

Inference helpers ready


**Reasoning**:
The inference helper functions have been re-executed with the corrected regex and key normalization. Now I will execute cell `52d3a8e6` to load validation data and set up the `generate_single` function, which is necessary before running batch inference.



In [55]:
from tqdm import tqdm

# Load validation data
with open(train_cfg["validation_dataset"], "r", encoding="utf-8") as f:
    eval_val_data = json.load(f)
print(f"Validation samples: {len(eval_val_data)}")

def generate_single(sample, max_new_tokens=2048):
    prompt = f"### Instruction:\n{sample['instruction']}\n\n### Input:\n{sample['input']}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

Validation samples: 20


**Reasoning**:
The inference helper functions and validation data setup are complete. Now I will run the batch inference on the validation dataset to collect results for evaluation.



In [56]:
print(f"Running inference on {len(eval_val_data)} validation samples...\n")

results = []
json_valid_count = 0
ats_valid_count = 0
predicted_scores = []
ground_truth_scores = []
predicted_skills = []
gt_skills = []

for i, sample in enumerate(tqdm(eval_val_data)):
    raw_output = generate_single(sample)
    parsed = extract_json(raw_output)
    gt = json.loads(sample["output"])

    is_valid_json = parsed is not None
    is_valid_ats = is_valid_json and validate_ats_output(parsed)

    if is_valid_json:
        json_valid_count += 1
    if is_valid_ats:
        ats_valid_count += 1
        predicted_scores.append(parsed["ats_score"])
        ground_truth_scores.append(gt["ats_score"])
        predicted_skills.append(set(parsed.get("missing_skills", [])))
        gt_skills.append(set(gt.get("missing_skills", [])))

    results.append({"index": i, "valid_json": is_valid_json, "valid_ats": is_valid_ats,
                     "predicted": parsed, "ground_truth": gt})

print("\nInference complete!")

Running inference on 20 validation samples...



  0%|          | 0/20 [00:25<?, ?it/s]
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_1703/1501991472.py", line 12, in <cell line: 0>
    raw_output = generate_single(sample)
                 ^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1703/3809582331.py", line 13, in generate_single
    outputs = model.generate(
              ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/peft/peft_model.py", line 2048, in generate
    outputs = self.base_model.generate(*args, **kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py", line 2669, in generate
    result = decoding_method(
       

TypeError: object of type 'NoneType' has no len()